# Predicción probabilística de fraude con tarjetas de crédito

**Informe final — LDS1071 · Kaggle Playground Series S6E11**

**Pregunta:** ¿qué representación, clasificador o ensamble maximiza la discriminación entre transacciones fraudulentas y legítimas? La respuesta es una probabilidad y la métrica principal es ROC AUC.

**Secuencia:** 1. EDA; 2. ingeniería de variables (K-means y DBSCAN); 3. clasificadores individuales y búsqueda anidada común; 4. ensambles (promedio, bagging, stacking con XGBoost y corrección logística de logits); 5. selección y evaluación; 6. conclusiones.

El modo nuevo entrena los ocho candidatos con los mismos cinco folds externos. El modo histórico conserva las comparaciones previas y no atribuye sus métricas al protocolo nuevo.


## 0. Configuración, fuentes y reproducibilidad

Trabajamos desde la raíz de `Credit-Fraud`. En Windows: `conda activate lds1071-fraud` y seleccione el kernel `Python (LDS1071 Fraud)`. El ZIP oficial `playground-series-s6e11.zip` debe estar en la raíz o los CSV en `data/`. Las figuras nuevas se guardan bajo `outputs/informe_final_es/figuras/` y **no se sobrescriben** los resultados originales.


In [ ]:
from __future__ import annotations
import os, zipfile, json, io, warnings, time, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, log_loss, brier_score_loss

SEMILLA = 42
EJECUTAR_PROTOCOLO_COMUN = False  # True: nuevo entrenamiento completo.
CARGAR_PROTOCOLO_COMUN = False  # True: reutiliza una ejecución completa de experiment021.
assert not (EJECUTAR_PROTOCOLO_COMUN and CARGAR_PROTOCOLO_COMUN)
N_PARTICIONES = 5
N_BAGGING = 5  # Réplicas bootstrap del XGBoost seleccionado.
assert N_BAGGING >= 2
MUESTRA_GRAFICAS = 20_000
RECALCULAR_BOOTSTRAP = False         # True: recalcula 600 réplicas si se dispone del OOF privado.
EJECUTAR_BUSQUEDA_HIBRIDA = False  # True: reentrena la búsqueda anidada del experimento018 (costosa; requiere script y datos).

RAIZ = Path.cwd()
if not (RAIZ / 'credit_card_fraud.ipynb').exists() and (RAIZ / 'final_project' / 'credit_card_fraud.ipynb').exists():
    RAIZ = RAIZ / 'final_project'
if not (RAIZ / 'credit_card_fraud.ipynb').exists():
    # Permitir ejecutar la versión descargada aislada junto a sus archivos de soporte.
    RAIZ = Path.cwd()
DATOS_AGREGADOS = RAIZ / 'experiments' / 'experiment016' / 'resultados'
if not DATOS_AGREGADOS.is_dir() and (RAIZ / 'resultados').is_dir():
    DATOS_AGREGADOS = RAIZ / 'resultados'
SALIDA = RAIZ / 'outputs' / ('informe_ensambles_es' if EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN else 'informe_final_es')
FIGURAS = SALIDA / 'figuras'
FIGURAS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette='dark', context='notebook')
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':150, 'axes.titlesize':12, 'axes.titleweight':'semibold'})
COLORES = sns.color_palette('dark')

def cerrar_figura(fig, nombre):
    fig.tight_layout()
    fig.savefig(FIGURAS / f'{nombre}.png', bbox_inches='tight', dpi=150)
    plt.show(); plt.close(fig)

DICCIONARIO = json.loads((DATOS_AGREGADOS/'diccionario_variables_es.json').read_text(encoding='utf-8')) if (DATOS_AGREGADOS/'diccionario_variables_es.json').exists() else {}
def etiqueta(col):
    return DICCIONARIO.get(col, str(col).replace('_',' ').capitalize())

print('Ubicación del proyecto:', RAIZ.resolve())
print('Modo:', 'Nuevo protocolo común' if EJECUTAR_PROTOCOLO_COMUN else ('Cargar protocolo común' if CARGAR_PROTOCOLO_COMUN else 'Reproducción histórica'))
print('Versiones: Python', sys.version.split()[0], '| pandas', pd.__version__)

def anotaciones_matriz(m):
    total=np.sum(m)
    return np.array([[f'{int(valor):,}\n({valor/total:.2%})' for valor in fila] for fila in m],dtype=object)

## 1. Análisis exploratorio de datos (EDA)

### 1.1 Variables, integridad y prevalencia de fraude

Se examinan tipos de variables, distribución del objetivo, faltantes y desbalance de clases antes de ajustar modelos. El identificador `id` se excluye de los predictores.


In [ ]:
def leer_csv(nombre):
    for ruta in [RAIZ/'data'/nombre, RAIZ/nombre, Path('/kaggle/input/playground-series-s6e11')/nombre]:
        if ruta.is_file():
            return pd.read_csv(ruta)
    for ruta in [RAIZ/'playground-series-s6e11.zip', RAIZ/'data'/'playground-series-s6e11.zip', RAIZ.parent/'playground-series-s6e11.zip']:
        if ruta.is_file():
            with zipfile.ZipFile(ruta) as archivo:
                with archivo.open(nombre) as fichero:
                    return pd.read_csv(fichero)
    raise FileNotFoundError(f'No se encontró {nombre}. Coloque el ZIP original de Kaggle en la raíz.')

entrenamiento = leer_csv('train.csv')
prueba = leer_csv('test.csv')
plantilla_envio = leer_csv('sample_submission.csv')
OBJETIVO, IDENTIFICADOR = 'is_fraud', 'id'
assert OBJETIVO not in prueba and plantilla_envio[IDENTIFICADOR].equals(prueba[IDENTIFICADOR])
assert entrenamiento[IDENTIFICADOR].is_unique and prueba[IDENTIFICADOR].is_unique
assert entrenamiento[OBJETIVO].notna().all() and set(entrenamiento[OBJETIVO].unique()) == {0.,1.}
variables = [c for c in entrenamiento.columns if c not in [OBJETIVO, IDENTIFICADOR]]
assert set(variables) == set(prueba.columns) - {IDENTIFICADOR}
numericas = entrenamiento[variables].select_dtypes(include='number').columns.tolist()
categoricas = entrenamiento[variables].select_dtypes(include=['object','category']).columns.tolist()
logicas = entrenamiento[variables].select_dtypes(include='bool').columns.tolist()
y = entrenamiento[OBJETIVO].astype(int).to_numpy()
assert len(variables)==24 and len(numericas)==13 and len(categoricas)==5 and len(logicas)==6
print(f'Entrenamiento: {len(entrenamiento):,} | Prueba: {len(prueba):,} | Variables: {len(variables)}')
print('Fraude:',int(y.sum()),f'({y.mean():.2%})')
display(entrenamiento.head(3))
resumen_variables = pd.DataFrame({'variable':variables,'nombre_en_español':[etiqueta(c) for c in variables],
 'tipo':[str(entrenamiento[c].dtype) for c in variables], 'faltantes':[int(entrenamiento[c].isna().sum()) for c in variables]})
display(resumen_variables)

### 1.2 Distribuciones y desbalance

La exploración utiliza totales completos para prevalencia, tasas por categoría e interacciones; en gráficos con muchos puntos se toma una muestra aleatoria de tamaño declarado, para evitar saturación. **No se elimina automáticamente ninguna observación extrema**: una transacción inusual puede contener información relevante para detectar fraude.

### 2.1 Desbalance, valores faltantes y distribución numérica


In [ ]:
muestra = entrenamiento.sample(n=min(MUESTRA_GRAFICAS,len(entrenamiento)), random_state=SEMILLA).copy()
fig, ejes = plt.subplots(1,2,figsize=(13,4.7))
conteos = entrenamiento[OBJETIVO].value_counts().reindex([0.,1.],fill_value=0)
sns.barplot(x=['Legítimas','Fraudulentas'],y=conteos.values,hue=['Legítimas','Fraudulentas'],
            palette=[COLORES[0],COLORES[3]],legend=False,ax=ejes[0])
for k,v in enumerate(conteos): ejes[0].text(k,v*1.01,f'{v:,} ({v/len(entrenamiento):.2%})',ha='center',fontsize=10)
ejes[0].set(title='Distribución observada de clases',xlabel='Clase',ylabel='Número de transacciones',ylim=(0,conteos.max()*1.14))
faltantes=entrenamiento[variables].isna().mean()*100
sns.barplot(x=faltantes.values,y=[etiqueta(c) for c in faltantes.index],color=COLORES[0],ax=ejes[1])
ejes[1].set(title='Datos faltantes por variable',xlabel='Porcentaje de valores faltantes',ylabel='Variable',xlim=(0,max(1,float(faltantes.max())*1.12)))
cerrar_figura(fig,'01_desbalance_y_faltantes')

cols=[c for c in ['amount_usd','hours_since_last_txn','distance_from_home_km','velocity_score','merchant_risk_score','account_balance_usd'] if c in muestra]
fig,ejes=plt.subplots(2,3,figsize=(16,8.8))
for ax,col in zip(ejes.ravel(),cols):
    sns.histplot(data=muestra,x=col,bins=42,color=COLORES[0],ax=ax)
    if col in ('amount_usd','account_balance_usd') and (muestra[col] > 0).all(): ax.set_xscale('log')
    ax.set(title=etiqueta(col),xlabel=etiqueta(col),ylabel='Transacciones (muestra)')
for ax in ejes.ravel()[len(cols):]: ax.axis('off')
fig.suptitle('Distribuciones de variables numéricas (muestra aleatoria)',y=1.01,fontweight='bold')
cerrar_figura(fig,'02_distribuciones_numericas')

### 1.3 Correlaciones y variables categóricas

La correlación de Pearson mide asociación lineal; Spearman evalúa ordenamientos monótonos. La ausencia de correlación alta **no** descarta interacciones ni dependencias no lineales. Las tasas por categoría se calculan sobre el **conjunto de entrenamiento completo**, no sobre la muestra empleada para visualizar.


In [ ]:
matriz=entrenamiento[numericas+logicas].astype(float)
fig,ax=plt.subplots(figsize=(12,10))
sns.heatmap(matriz.corr(method='spearman'),cmap='vlag',center=0,vmin=-1,vmax=1,
            xticklabels=[etiqueta(c) for c in matriz.columns],
            yticklabels=[etiqueta(c) for c in matriz.columns],ax=ax,
            cbar_kws={'label':'Correlación de Spearman'})
ax.set_title('Asociación monótona entre variables numéricas y lógicas')
cerrar_figura(fig,'03_correlacion_spearman')

fig,ejes=plt.subplots(1,2,figsize=(15.3,5.8))
for ax,col in zip(ejes,['merchant_category','channel']):
    g=entrenamiento.groupby(col,dropna=False)[OBJETIVO].agg(['mean','size']).sort_values('mean',ascending=False)
    g['nombre']=[f'{cat} (n={n:,})' for cat,n in zip(g.index,g['size'])]
    sns.barplot(data=g,x='mean',y='nombre',color=COLORES[0],ax=ax)
    ax.axvline(np.mean(y),linestyle='--',color=COLORES[3],label='Prevalencia total')
    ax.set(xlabel='Proporción observada de fraude',ylabel='',title=f'Riesgo por {etiqueta(col).lower()}')
    ax.legend(title='Referencia')
fig.suptitle('Tasas de fraude según categoría y canal (entrenamiento completo)',y=1.01,fontweight='bold')
cerrar_figura(fig,'04_fraude_por_categorias')

### 1.4 No linealidades e interacciones

Una transformación que retenga el 90 % de la varianza no necesariamente preservará los contrastes asociados al fraude. Antes de modelar evaluamos tasas de fraude por cuantiles para identificar posibles umbrales y relaciones no monótonas. Las tasas marginales no son efectos causales ni sustituyen al ajuste multivariante.


In [ ]:
analizar=['amount_usd','hours_since_last_txn','velocity_score','merchant_risk_score']
fig,ejes=plt.subplots(2,2,figsize=(14.4,9))
for ax,col in zip(ejes.ravel(),analizar):
    intervalos=pd.qcut(entrenamiento[col],q=12,duplicates='drop')
    tabla=entrenamiento.groupby(intervalos,observed=True)[OBJETIVO].agg(['mean','size']).reset_index()
    sns.lineplot(x=np.arange(1,len(tabla)+1),y=100*tabla['mean'],marker='o',color=COLORES[0],ax=ax)
    ax.axhline(100*np.mean(y),linestyle='--',color=COLORES[3],label='Prevalencia total')
    ax.set(title=etiqueta(col),xlabel='Intervalo por cuantiles (de menor a mayor)',ylabel='Operaciones fraudulentas (%)')
    ax.legend()
fig.suptitle('Relaciones marginales no lineales con el fraude',fontweight='bold',y=1.01)
cerrar_figura(fig,'05_tasas_por_cuantiles')

pares=[('merchant_risk_score','velocity_score'),('amount_usd','distance_from_home_km')]
fig,ejes=plt.subplots(1,2,figsize=(15,6.1))
for ax,(a,b) in zip(ejes,pares):
    ga=pd.qcut(entrenamiento[a],q=5,labels=False,duplicates='drop')
    gb=pd.qcut(entrenamiento[b],q=5,labels=False,duplicates='drop')
    t=entrenamiento.groupby([ga.rename('a'),gb.rename('b')])[OBJETIVO].mean().unstack()*100
    sns.heatmap(t,annot=True,fmt='.2f',cmap='crest',ax=ax,cbar_kws={'label':'Fraude observado (%)'})
    ax.set(xlabel=f'{etiqueta(b)} (quintil)',ylabel=f'{etiqueta(a)} (quintil)',title='Tasa conjunta por quintiles')
fig.suptitle('Interacciones bivariantes en el conjunto completo',fontweight='bold',y=1.01)
cerrar_figura(fig,'06_interacciones_quintiles')

### 1.5 Patrones temporales y anomalías

Una hora del día es una variable periódica: 23:00 y 00:00 están próximas pese a tener valores numéricos distantes. Se consideran transformaciones trigonométricas como posibilidad de ingeniería de variables. Para los extremos se emplean cercas de 3×IQR únicamente como **diagnóstico descriptivo**, junto con un detector no supervisado (Isolation Forest) sobre una muestra.


In [ ]:
fig,ejes=plt.subplots(1,2,figsize=(14,4.8))
for ax,col in zip(ejes,['time_of_day_hour','day_of_week']):
    tasa=entrenamiento.groupby(col)[OBJETIVO].agg(['mean','size'])
    sns.barplot(x=tasa.index,y=100*tasa['mean'],color=COLORES[0],ax=ax)
    ax.axhline(100*np.mean(y),linestyle='--',color=COLORES[3],label='Prevalencia total')
    ax.set(xlabel=etiqueta(col),ylabel='Porcentaje de fraude',title='Variación temporal del riesgo')
    ax.legend()
cerrar_figura(fig,'07_variacion_temporal')

extremos=[]
for c in numericas:
    q1,q3=entrenamiento[c].quantile([.25,.75]); iq=q3-q1
    if iq==0:continue
    flag=(entrenamiento[c]<q1-3*iq)|(entrenamiento[c]>q3+3*iq)
    extremos.append({'Variable':etiqueta(c),'Proporción extrema':float(flag.mean()),'Tasa de fraude entre extremos':float(entrenamiento.loc[flag,OBJETIVO].mean()) if flag.any() else np.nan})
tabla_extremos=pd.DataFrame(extremos).sort_values('Proporción extrema',ascending=False)
fig,ax=plt.subplots(figsize=(12,6))
sns.barplot(data=tabla_extremos.head(9),x='Proporción extrema',y='Variable',color=COLORES[3],ax=ax)
ax.set(title='Valores fuera de las cercas conservadoras de 3×IQR',xlabel='Proporción de operaciones',ylabel='Variable')
cerrar_figura(fig,'08_extremos_univariados')

if len(muestra)>500:
    base_anom=muestra[numericas+logicas].astype(float).replace([np.inf,-np.inf],np.nan)
    base_anom=base_anom.fillna(base_anom.median())
    scaler_anom=RobustScaler().fit_transform(base_anom)
    iso=IsolationForest(n_estimators=90,max_samples=256,contamination=0.02,random_state=SEMILLA,n_jobs=-1)
    score=-iso.fit(scaler_anom).decision_function(scaler_anom)
    m=muestra[[OBJETIVO]].copy();m['índice de anomalía']=score
    m['clase']=m[OBJETIVO].map({0.:'Legítima',1.:'Fraudulenta'})
    fig,ax=plt.subplots(figsize=(11,4.7))
    sns.histplot(data=m,x='índice de anomalía',hue='clase',stat='density',common_norm=False,
                 bins=40,element='step',fill=False,ax=ax,palette={'Legítima':COLORES[0],'Fraudulenta':COLORES[3]})
    ax.set(title='Anomalía multivariante: distribución según clase (muestra)',xlabel='Puntuación de anomalía (mayor = más atípica)',ylabel='Densidad por clase')
    cerrar_figura(fig,'09_anomalias_isolation_forest')

### 1.6 Comparación de entrenamiento y prueba

Como las etiquetas de la prueba son desconocidas, solo se comparan las distribuciones de los predictores. Una diferencia de distribuciones entre `train` y `test` podría comprometer la generalización; **no** implica que exista una etiqueta de fraude disponible en la prueba.


In [ ]:
variables_traspaso=['amount_usd','velocity_score','merchant_risk_score']
fig,ejes=plt.subplots(1,3,figsize=(16,4.7))
for ax,c in zip(ejes,variables_traspaso):
    a=entrenamiento[c].sample(n=min(12000,len(entrenamiento)),random_state=SEMILLA).reset_index(drop=True)
    b=prueba[c].sample(n=min(12000,len(prueba)),random_state=SEMILLA).reset_index(drop=True)
    sns.ecdfplot(a,ax=ax,color=COLORES[0],label='Entrenamiento')
    sns.ecdfplot(b,ax=ax,color=COLORES[1],label='Prueba')
    ax.set(title=etiqueta(c),xlabel=etiqueta(c),ylabel='Proporción acumulada')
    ax.legend()
fig.suptitle('Comparación no supervisada de las distribuciones',fontweight='bold',y=1.01)
cerrar_figura(fig,'10_desplazamiento_entrenamiento_prueba')

## 2. Ingeniería de variables: PCA, K-means y DBSCAN

Las características derivadas de agrupamientos son **variables nuevas, no transacciones sintéticas**. Su construcción, sus hiperparámetros y su escalado deben ajustarse exclusivamente con observaciones de entrenamiento en cada partición de validación cruzada.

PCA/SVD y t-SNE permiten explorar estructura, pero la visualización t-SNE no se introduce como predictor de la prueba. K-means aporta distancias, pertenencias y similitud a centroides; DBSCAN aporta proximidad a núcleos y medidas locales de densidad.


In [ ]:
# PCA solamente ilustrativa sobre una muestra de entrenamiento: no interviene en ninguna predicción OOF.
X_visual=muestra[variables]
preprocesador_visual=ColumnTransformer([
    ('num',Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler())]),numericas+logicas),
    ('cat',Pipeline([('imputar',SimpleImputer(strategy='most_frequent')),
                     ('codificar',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categoricas),
],sparse_threshold=0)
M=preprocesador_visual.fit_transform(X_visual)
pca_v=PCA(n_components=.90,svd_solver='full',random_state=SEMILLA).fit(M)
pca_total=PCA(n_components=min(30,M.shape[1]),svd_solver='full',random_state=SEMILLA).fit(M)
fig,ejes=plt.subplots(1,2,figsize=(14.5,4.8))
ejes[0].plot(np.arange(1,len(pca_total.explained_variance_ratio_)+1),pca_total.explained_variance_ratio_.cumsum(),marker='o',markersize=3,color=COLORES[0])
ejes[0].axhline(.90,ls='--',color=COLORES[3],label='90 % de varianza')
ejes[0].set(xlabel='Número de componentes',ylabel='Varianza acumulada',title='PCA ilustrativa: varianza explicada')
ejes[0].legend()
XY=PCA(n_components=2,random_state=SEMILLA).fit_transform(M)
seleccion=np.random.default_rng(SEMILLA).choice(len(XY),size=min(5000,len(XY)),replace=False)
clases=muestra.iloc[seleccion][OBJETIVO].values.astype(int)
for val,color,nombre in [(0,COLORES[0],'Legítima'),(1,COLORES[3],'Fraudulenta')]:
    mask=clases==val
    ejes[1].scatter(XY[seleccion[mask],0],XY[seleccion[mask],1],s=8,alpha=.48,color=color,label=nombre,rasterized=True)
ejes[1].set(xlabel='Primera componente',ylabel='Segunda componente',title='Proyección PCA en dos dimensiones')
ejes[1].legend(title='Clase observada')
fig.suptitle('Reducción dimensional para explorar la estructura de los predictores',fontweight='bold',y=1.01)
cerrar_figura(fig,'11_pca_90_varianza')
print('PCA ilustrativa: componentes para 90 % de varianza:',pca_v.n_components_,'de',M.shape[1])

### 2.1 t-SNE: visualización de vecindades

Se proyectan datos preprocesados en dos dimensiones mediante t-SNE. La representación de los fraudes se enriquece exclusivamente para que sean visibles en la gráfica; **no** es la prevalencia real y las coordenadas no se usan para clasificar. PCA y la geometría de clustering son transformaciones diferentes, ajustadas dentro de entrenamiento para prevenir fuga de información.


In [ ]:
from sklearn.manifold import TSNE
generador_tsne=np.random.default_rng(SEMILLA)
pos=np.flatnonzero(muestra[OBJETIVO].to_numpy()==1)
neg=np.flatnonzero(muestra[OBJETIVO].to_numpy()==0)
idx=np.r_[generador_tsne.choice(neg,size=min(1500,len(neg)),replace=False),
          generador_tsne.choice(pos,size=min(270,len(pos)),replace=False)]
coords=TSNE(n_components=2,perplexity=35,init='pca',learning_rate='auto',max_iter=800,random_state=SEMILLA).fit_transform(M[idx])
clases=muestra.iloc[idx][OBJETIVO].to_numpy()
fig,ax=plt.subplots(figsize=(9,5.6))
for clase,color,nombre in [(0,COLORES[0],'Legítima'),(1,COLORES[3],'Fraudulenta')]:
    mask=clases==clase
    ax.scatter(coords[mask,0],coords[mask,1],color=color,s=15,alpha=.5,label=f'{nombre} (n={mask.sum():,})',rasterized=True)
ax.set(title='t-SNE: vecindades locales (muestra enriquecida)',xlabel='Coordenada t-SNE 1',ylabel='Coordenada t-SNE 2')
ax.legend(title='Clase observada')
cerrar_figura(fig,'11b_tsne')

### 2.2 K-means y DBSCAN: diagnósticos antes de entrenar los clasificadores

Se comparan inercia y silueta para K-means, y radio de vecindad, ruido y número de grupos para DBSCAN. Los parámetros de la geometría se escogen **dentro del entrenamiento de cada fold**, no con las observaciones destinadas a evaluar el clasificador.

La construcción geométrica utiliza muestras de referencia de entrenamiento (15 000 para K-means/PCA y 2 300 para DBSCAN); los modelos supervisados se entrenan con todas las filas disponibles de su fold.


In [ ]:
ruta_diag=DATOS_AGREGADOS/'experiment013_cluster_diagnostics.csv'
if ruta_diag.exists():
    diagnosticos=pd.read_csv(ruta_diag)
    diag_k=diagnosticos[(diagnosticos['algorithm']=='KMeans')]
    diag_db=diagnosticos[(diagnosticos['algorithm']=='DBSCAN')]
    fig,ejes=plt.subplots(1,3,figsize=(16.5,4.7))
    sns.lineplot(data=diag_k,x='k',y='silhouette',hue='fold',marker='o',palette='dark',ax=ejes[0])
    ejes[0].set(title='K-means: índice de silueta',xlabel='Número de clusters (k)',ylabel='Silueta')
    ejes[0].legend(title='Pliegue',fontsize=8)
    sns.lineplot(data=diag_k,x='k',y='inertia',hue='fold',marker='o',palette='dark',ax=ejes[1])
    ejes[1].set(title='K-means: inercia',xlabel='Número de clusters (k)',ylabel='Inercia total')
    ejes[1].legend(title='Pliegue',fontsize=8)
    sns.lineplot(data=diag_db,x='param',y='noise_fraction',hue='fold',marker='o',palette='dark',ax=ejes[2])
    ejes[2].set(title='DBSCAN: proporción identificada como ruido',xlabel='Radio de vecindad (ε)',ylabel='Proporción de ruido')
    ejes[2].legend(title='Pliegue',fontsize=8)
    fig.suptitle('Diagnóstico no supervisado dentro de los entrenamientos por pliegue',y=1.02,fontweight='bold')
    cerrar_figura(fig,'12b_seleccion_clusters')
else:
    print('El archivo de diagnósticos de clusters no está disponible.')

### 2.3 Representaciones candidatas

A partir de la matriz preprocesada se distinguen cuatro alternativas: **original**, **original + K-means**, **original + DBSCAN** y **original + ambos**. Las columnas originales no se eliminan al añadir agrupamientos. Las distancias a centroides o núcleos permiten transformar una observación nueva sin volver a ajustar los clusters.

La variante fija de cinco centroides se estudia también como control. El efecto predictivo de cada representación se evalúa en la sección 3, después de especificar los clasificadores y la validación.


## 3. Clasificadores individuales y búsqueda anidada común

Todas las familias utilizan las mismas cinco particiones externas, la misma muestra interna estratificada de hasta 36 000 transacciones por fold y las mismas tres particiones internas. Se seleccionan representación e hiperparámetros maximizando la media de ROC AUC interna y se reajusta el ganador con todo el entrenamiento externo.

XGBoost individual utiliza **36 configuraciones por representación**, es decir, 144 configuraciones por fold al evaluar las cuatro representaciones. La logística también prueba PCA y splines. Los resultados históricos se muestran únicamente en modo histórico.


### 3.1 Familias de clasificadores y supuestos

- **Regresión logística:** $\operatorname{logit}(p)=\beta_0+x^\top\beta$; se ajusta regularización L2 y ponderación. Splines permiten efectos aditivos no lineales; PCA comprime la matriz de predictores.
- **Bosque aleatorio:** promedio de árboles aleatorizados; profundidad y tamaño mínimo de hoja controlan complejidad.
- **XGBoost:** suma secuencial de árboles que minimiza una pérdida logística regularizada.
- **CatBoost:** potenciación de árboles; el ajuste histórico conserva su tratamiento nativo de categorías. El protocolo común utiliza la misma matriz codificada para todas las familias, por lo que su CatBoost es una variante nueva.

La logística no exige normalidad de los predictores. Se revisan sus supuestos porque será la base del modelo secuencial.


In [ ]:
supuestos=pd.DataFrame([
    ['Respuesta binaria','Verificado','La etiqueta toma los valores 0 y 1.'],
    ['Predictores observados / faltantes','Verificado','Se auditan los faltantes y la imputación se ajusta por pliegue.'],
    ['Linealidad en el logit','No garantizada','Se detectaron patrones no lineales; se contrastaron splines y árboles.'],
    ['Independencia por cliente','No verificable','No se proporciona un identificador de cliente longitudinal.'],
    ['Colinealidad / redundancia','Mitigada','Categorías con nivel de referencia y regularización L2.'],
    ['Separación completa','No demostrada ausente','Una revisión univariada no descarta separación multivariada.'],
    ['Tamaño de muestra','Adecuado para empezar','Hay 11 689 fraudes; importa también la rareza de combinaciones de categorías.'],
    ['Calibración de probabilidades','Se evalúa con OOF','Pérdida logarítmica, Brier y curvas de calibración.'],
],columns=['Condición','Situación','Evidencia y limitación'])
display(supuestos)

### 3.2 Espacios de búsqueda

Cada familia tiene sus propios parámetros. XGBoost individual, el meta-modelo XGBoost de stacking y el corrector de logits comparten exactamente la misma grilla: profundidad ∈ {3, 5, 7, 10, 15, 30}, tasa de aprendizaje ∈ {0.04, 0.08}, peso mínimo del hijo ∈ {5, 10, 15}. Son 36 combinaciones; se mantienen también los mismos 240 árboles, subsample y colsample.

El bagging reutiliza los parámetros elegidos para XGBoost individual y añade cinco réplicas bootstrap; el promedio de familias no requiere una búsqueda adicional.


In [ ]:
from sklearn.model_selection import ParameterGrid
from sklearn.neighbors import NearestNeighbors
from sklearn.base import clone

def preprocesador_numerico_y_categorico(escalar=True):
    pasos=[('imputar',SimpleImputer(strategy='median'))]
    if escalar: pasos.append(('escalar',StandardScaler()))
    return ColumnTransformer([
        ('numeros',Pipeline(pasos),numericas+logicas),
        ('categorias',Pipeline([('imputar',SimpleImputer(strategy='most_frequent')),
                               ('codificar',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categoricas),
    ],sparse_threshold=0)

REPRESENTACIONES = ('baseline','kmeans','dbscan','combined')
GRILLAS_COMUNES = {
    'Logística': {'C':[.01,.1,1.,10.], 'class_weight':[None,'balanced']},
    'Bosque aleatorio': {'max_depth':[12,18,None], 'min_samples_leaf':[5,10,15]},
    'XGBoost': {'max_depth':[3,5,7,10,15,30], 'learning_rate':[.04,.08], 'min_child_weight':[5,10,15]},
    'CatBoost': {'depth':[4,6], 'learning_rate':[.04,.08], 'l2_leaf_reg':[3.,7.]},
}
FAMILIAS_INDIVIDUALES = tuple(GRILLAS_COMUNES)
GRILLA_BOOSTER = GRILLAS_COMUNES['XGBoost']
GRILLAS_COMUNES['Stacking XGBoost'] = GRILLA_BOOSTER
GRILLAS_COMUNES['Logística + XGBoost'] = GRILLA_BOOSTER
ENSAMBLES = ('Promedio de familias','Bagging XGBoost','Stacking XGBoost','Logística + XGBoost')
CANDIDATOS = FAMILIAS_INDIVIDUALES + ENSAMBLES
assert len(ParameterGrid(GRILLA_BOOSTER)) == 36
def representaciones_familia(familia):
    return REPRESENTACIONES + ('pca90','splines') if familia=='Logística' else REPRESENTACIONES

tabla_grillas=[]
for familia,grilla in GRILLAS_COMUNES.items():
    reps=representaciones_familia(familia) if familia in FAMILIAS_INDIVIDUALES else (('probabilidades OOF de cuatro familias',) if familia=='Stacking XGBoost' else ('base logística seleccionada',))
    tabla_grillas.append({'Familia':familia,'Representaciones':', '.join(reps),
                          'Grilla':str(grilla),'Combinaciones por representación':len(ParameterGrid(grilla)),
                          'Combinaciones por fold':len(ParameterGrid(grilla))*len(reps)})
display(pd.DataFrame(tabla_grillas))
display(pd.DataFrame([{'Familia':'Bagging XGBoost','Configuración':'XGBoost individual seleccionado; 5 réplicas bootstrap'}, {'Familia':'Promedio de familias','Configuración':'Media uniforme de las cuatro probabilidades'}]))


### 3.3 Algoritmo de búsqueda anidada

Para cada fold externo se crean tres splits internos y se ajusta cada transformación únicamente con su entrenamiento. Se registran todas las combinaciones, sus puntuaciones y la configuración ganadora. Se conservan las probabilidades de validación y prueba de los cuatro modelos individuales **por fold**, para construir ensambles sin utilizar predicciones de un fold que haya visto sus etiquetas.


In [ ]:
def matrices_por_split(x_ent, otros):
    pre=preprocesador_numerico_y_categorico()
    a=pre.fit_transform(x_ent).astype(np.float32)
    b=[pre.transform(x).astype(np.float32) for x in otros]
    rng=np.random.default_rng(SEMILLA)
    ref=a[rng.choice(len(a),min(15000,len(a)),replace=False)]
    pca=PCA(n_components=.90,svd_solver='full').fit(ref)
    z=[pca.transform(x) for x in [a]+b]
    zref=pca.transform(ref)
    km=KMeans(n_clusters=5,n_init=10,random_state=SEMILLA).fit(zref)
    refdb=zref[rng.choice(len(zref),min(2300,len(zref)),replace=False)]
    vecinos=NearestNeighbors(n_neighbors=5).fit(refdb)
    eps=max(float(np.quantile(vecinos.kneighbors(refdb)[0][:,-1],.90)),1e-6)
    db=DBSCAN(eps=eps,min_samples=5).fit(refdb)
    nucleos=refdb[db.core_sample_indices_]
    nn=NearestNeighbors(n_neighbors=1).fit(nucleos) if len(nucleos) else None
    spline_cols=[(numericas+logicas).index(c) for c in ('amount_usd','velocity_score','merchant_risk_score') if c in numericas+logicas]
    spline=SplineTransformer(n_knots=5,degree=3,include_bias=False,extrapolation='constant').fit(a[:,spline_cols])
    resto=[j for j in range(a.shape[1]) if j not in spline_cols]
    matrices=[]
    for x,zz in zip([a]+b,z):
        dk=km.transform(zz)
        if nn is None:
            dd=np.full((len(x),1),2*eps)
        else:
            dd=nn.kneighbors(zz,return_distance=True)[0]
        kd=np.c_[dk,dk.min(axis=1)]
        bd=np.c_[dd,(dd>eps).astype(float)]
        matrices.append({'baseline':x,'kmeans':np.c_[x,kd].astype(np.float32),
                         'dbscan':np.c_[x,bd].astype(np.float32),
                         'combined':np.c_[x,kd,bd].astype(np.float32),
                         'pca90':zz.astype(np.float32),
                         'splines':np.c_[x[:,resto],spline.transform(x[:,spline_cols])].astype(np.float32)})
    # Escalar columnas derivadas con estadísticas del entrenamiento; mismo tratamiento para todas las familias.
    for nombre in matrices[0]:
        if nombre=='baseline': continue
        scaler=StandardScaler().fit(matrices[0][nombre])
        for m in matrices: m[nombre]=scaler.transform(m[nombre]).astype(np.float32)
    return matrices

def nuevo_modelo(familia,parametros):
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier
    if familia=='Logística':
        return LogisticRegression(**parametros,max_iter=600,solver='lbfgs',random_state=SEMILLA)
    if familia=='Bosque aleatorio':
        return RandomForestClassifier(**parametros,n_estimators=240,n_jobs=-1,random_state=SEMILLA)
    if familia=='CatBoost':
        return CatBoostClassifier(**parametros,iterations=240,task_type='CPU',thread_count=4,verbose=False,
                                  allow_writing_files=False,random_seed=SEMILLA)
    return XGBClassifier(**parametros,n_estimators=240,subsample=.85,colsample_bytree=.85,
                         objective='binary:logistic',eval_metric='logloss',tree_method='hist',
                         device='cpu',n_jobs=4,random_state=SEMILLA)

def probabilidades(modelo,x):
    return np.asarray(modelo.predict_proba(x)[:,1],dtype=float)

def splits_externos(X,y):
    return list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEMILLA).split(X,y))

def muestra_interna(tr,y,fold):
    if len(tr)<=36000:
        return tr
    sub,_=train_test_split(tr,train_size=36000,stratify=y[tr],random_state=SEMILLA+fold)
    return sub

def splits_internos(sub,y,fold):
    return list(StratifiedKFold(n_splits=3,shuffle=True,random_state=SEMILLA+100+fold).split(sub,y[sub]))

def cache_interna(X,y,sub,fold):
    cache=[]
    for it,iv in splits_internos(sub,y,fold):
        a,b=matrices_por_split(X.iloc[sub[it]],[X.iloc[sub[iv]]])
        cache.append((a,b,y[sub[it]],y[sub[iv]]))
    return cache

def registro_grilla(registros,fold,familia,rep,par,scores):
    media=float(np.mean(scores))
    registros.append(dict(fold=fold,familia=familia,representacion=rep,parametros=json.dumps(par,sort_keys=True),
                          auc_interna=media,sd_interna=float(np.std(scores))))
    return media

def registrar_ganador(ganadores,fold,familia,rep,par,auc,yv,pv):
    ganadores.append(dict(fold=fold,familia=familia,representacion=rep,parametros=json.dumps(par,sort_keys=True),
                          auc_interna=auc,auc_externa=roc_auc_score(yv,pv)))

def nuevas_predicciones(nombres,n,m):
    return {f:{'oof':np.full(n,np.nan),'test':np.zeros(m)} for f in nombres}

def agregar_prediccion(resultados,f,va,pv,pt):
    assert np.isfinite(pv).all() and np.isfinite(pt).all()
    assert ((pv>=0)&(pv<=1)).all() and ((pt>=0)&(pt<=1)).all()
    resultados[f]['oof'][va]=pv
    resultados[f]['test']+=pt/5

def resumir_predicciones(resultados,y):
    filas=[]
    for f,d in resultados.items():
        p=d['oof']
        assert p.shape==y.shape and np.isfinite(p).all() and np.isfinite(d['test']).all()
        filas.append(dict(familia=f,roc_auc=roc_auc_score(y,p),average_precision=average_precision_score(y,p),
                          log_loss=log_loss(y,p),brier=brier_score_loss(y,p)))
    return pd.DataFrame(filas).sort_values('roc_auc',ascending=False).reset_index(drop=True)

def protocolo_individual(X,y,X_test):
    y=np.asarray(y,dtype=int)
    resultados=nuevas_predicciones(FAMILIAS_INDIVIDUALES,len(y),len(X_test))
    registros=[]; ganadores=[]; estados=[]
    for fold,(tr,va) in enumerate(splits_externos(X,y),1):
        sub=muestra_interna(tr,y,fold)
        cache=cache_interna(X,y,sub,fold)
        elegidos={}
        for familia in FAMILIAS_INDIVIDUALES:
            opciones=[]
            for rep in representaciones_familia(familia):
                for par in ParameterGrid(GRILLAS_COMUNES[familia]):
                    scores=[]
                    for a,b,yt,yv in cache:
                        modelo=nuevo_modelo(familia,par).fit(a[rep],yt)
                        scores.append(roc_auc_score(yv,probabilidades(modelo,b[rep])))
                    media=registro_grilla(registros,fold,familia,rep,par,scores)
                    opciones.append((media,rep,par))
            elegidos[familia]=max(opciones,key=lambda q:q[0])
            print('Fold',fold,'individual',familia,'mejor AUC interna',round(elegidos[familia][0],6),flush=True)
        del cache
        a,b,c=matrices_por_split(X.iloc[tr],[X.iloc[va],X_test])
        pv_cols=[]; pt_cols=[]
        for familia in FAMILIAS_INDIVIDUALES:
            auc,rep,par=elegidos[familia]
            modelo=nuevo_modelo(familia,par).fit(a[rep],y[tr])
            pv=probabilidades(modelo,b[rep]); pt=probabilidades(modelo,c[rep])
            agregar_prediccion(resultados,familia,va,pv,pt)
            registrar_ganador(ganadores,fold,familia,rep,par,auc,y[va],pv)
            pv_cols.append(pv); pt_cols.append(pt)
        estados.append(dict(fold=fold,tr=tr,va=va,sub=sub,elegidos=elegidos,
                            meta_val=np.column_stack(pv_cols),meta_test=np.column_stack(pt_cols)))
        del a,b,c,modelo
    return resultados,pd.DataFrame(registros),pd.DataFrame(ganadores),estados

def mostrar_grillas(grilla,seleccion,familias,prefijo):
    for familia in familias:
        tabla=grilla.loc[grilla.familia==familia].copy()
        display(tabla.sort_values(['fold','auc_interna'],ascending=[True,False]).groupby('fold').head(3).round(6))
        display(seleccion.loc[seleccion.familia==familia].round(6))
        fig,ax=plt.subplots(figsize=(10,4.5))
        sns.boxplot(data=tabla,x='representacion',y='auc_interna',ax=ax,color=COLORES[0])
        ax.set(title=f'{familia}: distribución de AUC interna',xlabel='Representación',ylabel='ROC AUC interno')
        cerrar_figura(fig,prefijo+'_'+str(CANDIDATOS.index(familia)))

MODO_COMUN=EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN
directorio=RAIZ/'outputs'/'experiment021'
resultados_comunes={}
resultados_individuales={}
if EJECUTAR_PROTOCOLO_COMUN:
    resultados_individuales,grilla_individual,seleccion_individual,estados_individuales=protocolo_individual(entrenamiento[variables],y,prueba[variables])
elif CARGAR_PROTOCOLO_COMUN:
    resumen_comun=pd.read_csv(directorio/'summary.csv').sort_values('roc_auc',ascending=False).reset_index(drop=True)
    grilla_comun=pd.read_csv(directorio/'grid_results.csv')
    seleccion_comun=pd.read_csv(directorio/'selected_params.csv')
    assert set(resumen_comun.familia)==set(CANDIDATOS)
    with np.load(directorio/'predictions_PRIVATE.npz',allow_pickle=False) as z:
        assert np.array_equal(z['train_id'],entrenamiento[IDENTIFICADOR].to_numpy())
        assert np.array_equal(z['test_id'],prueba[IDENTIFICADOR].to_numpy())
        assert np.array_equal(z['y'],y)
        for f in CANDIDATOS:
            p=z[f'{f}__oof']; q=z[f'{f}__test']
            assert p.shape==(len(y),) and q.shape==(len(prueba),)
            assert np.isfinite(p).all() and np.isfinite(q).all()
            assert ((p>=0)&(p<=1)).all() and ((q>=0)&(q<=1)).all()
            assert np.isclose(roc_auc_score(y,p),resumen_comun.loc[resumen_comun.familia==f,'roc_auc'].iloc[0],atol=1e-10,rtol=0)
            resultados_comunes[f]={'oof':p,'test':q}
    resultados_individuales={f:resultados_comunes[f] for f in FAMILIAS_INDIVIDUALES}
    grilla_individual=grilla_comun[grilla_comun.familia.isin(FAMILIAS_INDIVIDUALES)]
    seleccion_individual=seleccion_comun[seleccion_comun.familia.isin(FAMILIAS_INDIVIDUALES)]
if resultados_individuales:
    resumen_individual=resumir_predicciones(resultados_individuales,y)
    display(resumen_individual.round(6))
    mostrar_grillas(grilla_individual,seleccion_individual,FAMILIAS_INDIVIDUALES,'grilla_individual')
    fig,ax=plt.subplots(figsize=(10,4.5))
    sns.scatterplot(data=resumen_individual,x='roc_auc',y='familia',s=90,ax=ax)
    ax.set(title='Clasificadores individuales: ROC AUC OOF',xlabel='ROC AUC OOF',ylabel='')
    cerrar_figura(fig,'individuales_comparacion')


### 3.4 Referencia histórica (solo modo histórico)

Las celdas siguientes no se ejecutan en el protocolo nuevo; se conservan para reproducir comparaciones previas.

In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    # Registro de configuraciones de la fase original; los nombres legibles se usan en las tablas.
    ESPECIFICACIONES = [
     ('logit_C0p1','Regresión logística','C=0.1'),
     ('logit_C1','Regresión logística','C=1'),
     ('logit_balanced_C1','Logística con ponderación de clases','C=1; clases equilibradas'),
     ('logit_spline_C0p1','Logística con splines','C=0.1'),
     ('logit_spline_C1','Logística con splines','C=1'),
     ('pca90_logit_C0p1','Logística con PCA','90 % de varianza; C=0.1'),
     ('pca90_logit_C1','Logística con PCA','90 % de varianza; C=1'),
     ('rf_depth12_leaf10','Bosque aleatorio','profundidad 12; hoja mínima 10'),
     ('rf_depth18_leaf5','Bosque aleatorio','profundidad 18; hoja mínima 5'),
     ('rf_depthNone_leaf15','Bosque aleatorio','profundidad libre; hoja mínima 15'),
     ('xgb_depth3_lr08','XGBoost individual','profundidad 3; tasa 0.08'),
     ('xgb_depth5_lr05','XGBoost individual','profundidad 5; tasa 0.05; 350 árboles'),
     ('xgb_depth6_lr04','XGBoost individual','profundidad 6; tasa 0.04'),
     ('catboost_depth5_lr06','CatBoost','profundidad 5; tasa 0.06'),
     ('catboost_depth7_lr04','CatBoost','profundidad 7; tasa 0.04'),
     ('offset_xgb_depth2_lr05','Logística corregida con XGBoost','profundidad de corrección 2; tasa 0.05'),
     ('offset_xgb_depth3_lr05','Logística corregida con XGBoost','profundidad de corrección 3; tasa 0.05; 240 árboles'),
     ('xgb_nested_grid','XGBoost ajustado con GridSearchCV','búsqueda interna de hiperparámetros'),
     ('catboost_nested_grid','CatBoost ajustado con GridSearchCV','búsqueda interna de hiperparámetros'),
    ]
    tabla_modelos=pd.DataFrame(ESPECIFICACIONES,columns=['Clave del experimento','Modelo','Configuración'])
    display(tabla_modelos)


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    ruta_grid=RAIZ/'outputs'/'nested_gridsearch_cv_results.csv'
    if ruta_grid.is_file():
        reg=pd.read_csv(ruta_grid)
        reg['Familia ajustada']=reg['candidate'].map({'xgb_nested_grid':'XGBoost ajustado','catboost_nested_grid':'CatBoost ajustado'}).fillna(reg['candidate'])
        fig,ax=plt.subplots(figsize=(11,5.3))
        sns.boxplot(data=reg,x='Familia ajustada',y='mean_test_roc_auc',color=COLORES[0],ax=ax)
        ax.set(title='Búsquedas históricas: ROC AUC interno',xlabel='Familia de modelos',ylabel='ROC AUC interno')
        cerrar_figura(fig,'13b_busqueda_hiperparametros')
        display(reg[['Familia ajustada','outer_fold','mean_test_roc_auc','std_test_roc_auc','rank_test_roc_auc']].round(6))
    else:
        print('Registro histórico detallado no disponible; las grillas originales están en credit_card_fraud.ipynb (sección GridSearchCV).')
    display(pd.DataFrame([
        ['Logística / splines / PCA','Configuraciones fijas','C=.1, 1; ponderación en un control','5 externos'],
        ['Bosque aleatorio','Configuraciones fijas','(profundidad, hoja): (12,10), (18,5), (libre,15)','5 externos'],
        ['XGBoost','GridSearchCV histórico','depth=[3,5]; eta=[.045,.10]; child=[3,9]','5 externos / 3 internos / 20k'],
        ['CatBoost','GridSearchCV histórico','depth=[4,6]; eta=[.045,.10]; L2=[3,9]','5 externos / 3 internos / 20k'],
        ['XGBoost + K-means 5','Grilla ampliada','36 combinaciones (experimento017)','5 externos / 3 internos'],
        ['Logística + XGBoost','Grilla ampliada','36 combinaciones × 4 representaciones (experimento018)','5 externos / 3 internos / 36k'],
    ],columns=['Familia','Diseño histórico','Espacio evaluado','Validación']))



In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    archivos_candidatos=[RAIZ/'outputs'/'candidate_metrics.csv',RAIZ.parent/'outputs'/'candidate_metrics.csv']
    archivo_candidatos=next((p for p in archivos_candidatos if p.is_file()),None)
    NOMBRES_CANDIDATO=dict((a,f'{b} · {c}') for a,b,c in ESPECIFICACIONES)
    if archivo_candidatos:
        candidatos_originales=pd.read_csv(archivo_candidatos)
        if {'candidate','roc_auc'}.issubset(candidatos_originales.columns):
            candidatos_originales['Modelo legible']=candidatos_originales['candidate'].map(NOMBRES_CANDIDATO).fillna(candidatos_originales['candidate'])
            display(candidatos_originales[['Modelo legible','roc_auc','average_precision','log_loss','brier']].sort_values('roc_auc',ascending=False).round(6))
            top=candidatos_originales.sort_values('roc_auc').tail(19).copy()
            top['Nombre corto']=top['Modelo legible'].str.replace(' · búsqueda interna de hiperparámetros',' (grilla)',regex=False)
            fig,ax=plt.subplots(figsize=(13.8,max(7.5,.46*len(top)+1.8)))
            posiciones=np.arange(len(top))
            ax.scatter(top['roc_auc'],posiciones,color=COLORES[0],s=55,zorder=4)
            mejor=float(top['roc_auc'].max())
            ax.axvline(mejor,color=COLORES[3],ls='--',lw=1.3,label=f'Mejor valor: {mejor:.6f}')
            menor=float(top['roc_auc'].min())
            margen=max((mejor-menor)*.065,.0006)
            ax.set_xlim(menor-margen,mejor+2.2*margen)
            ax.set_yticks(posiciones,top['Nombre corto'])
            for yy,valor in zip(posiciones,top['roc_auc']):
                ax.annotate(f'{valor:.6f}',(valor,yy),xytext=(6,0),textcoords='offset points',va='center',fontsize=8)
            ax.set(title='Comparación de candidatos por ROC AUC OOF',xlabel='ROC AUC fuera de partición (eje ampliado)',ylabel='Configuración')
            ax.grid(axis='x',alpha=.35)
            ax.legend(loc='lower right',fontsize=9)
            cerrar_figura(fig,'13_comparacion_candidatos')
    else:
        print('Sin candidate_metrics.csv: no se inventan resultados de candidatos.')


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    resumen_ablasiones=[]
    for e,modelo in [(13,'XGBoost individual'),(14,'Regresión logística'),(15,'Regresión logística + XGBoost')]:
        p=DATOS_AGREGADOS/f'experiment{e:03d}_summary.csv'
        if p.is_file():
            d=pd.read_csv(p)
            d['modelo']=modelo;d['experimento']=e
            resumen_ablasiones.append(d)
    if resumen_ablasiones:
        ablasiones=pd.concat(resumen_ablasiones,ignore_index=True)
        campos=['modelo','variant','roc_auc','delta_auc_vs_baseline']
        display(ablasiones[[c for c in campos if c in ablasiones.columns]].round(6))
        fig,ax=plt.subplots(figsize=(12,6.6))
        tabla=ablasiones[ablasiones['variant']!='baseline'].copy()
        trad={'kmeans':'K-means','dbscan':'DBSCAN','combined':'Ambos','combined_lr_only':'Ambos, solo en la logística'}
        tabla['Variables nuevas']=tabla['variant'].map(trad).fillna(tabla['variant'])
        tabla['Modelo']=tabla['modelo']
        sns.barplot(data=tabla,y='Modelo',x='delta_auc_vs_baseline',hue='Variables nuevas',ax=ax)
        ax.axvline(0,color='black',lw=1)
        ax.set(title='Cambio de ROC AUC tras incorporar variables de agrupamiento',xlabel='Diferencia respecto del modelo sin agrupamientos',ylabel='Familia evaluada')
        ax.legend(title='Características añadidas',bbox_to_anchor=(1.01,1),loc='upper left')
        cerrar_figura(fig,'12_ablasiones_agrupamiento')
    else:
        print('No se encontraron tablas agregadas de los experimentos 013–015.')


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    fig,ax=plt.subplots(figsize=(13.5,5.5))
    resumen=pd.DataFrame([
        ['XGBoost individual',0.9457023626,'Sin nuevas variables'],
        ['XGBoost + DBSCAN',0.9457050194,'DBSCAN'],
        ['Logística individual',0.9413568422,'Sin nuevas variables'],
        ['Logística + K-means y DBSCAN',0.9415591273,'Ambos'],
        ['Logística + XGBoost',0.9461023509,'Sin nuevas variables'],
        ['Logística con clusters + XGBoost original',0.9461606241,'Clusters solo en logística'],
    ],columns=['Modelo','ROC AUC','Ingeniería adicional'])
    sns.barplot(data=resumen,x='ROC AUC',y='Modelo',hue='Ingeniería adicional',ax=ax)
    ax.set(title='Ablaciones completas de agrupamientos: experimentos 013–015',xlabel='ROC AUC OOF',ylabel='Arquitectura')
    ax.legend(title='Variables nuevas',loc='lower right')
    cerrar_figura(fig,'18_ablasiones_resumen_final')


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    ruta_exp17=RAIZ/'outputs'/'experiment017'/'full_699960'
    ruta_comp=RAIZ/'outputs'/'experiment017'/'depth30'/'summary_expanded.csv'
    if not ruta_comp.is_file():
        ruta_comp=RAIZ/'experiments'/'experiment017'/'results'/'summary_kmeans5.csv'
    if ruta_comp.is_file():
        resultados_017=pd.read_csv(ruta_comp)
        if 'variant' not in resultados_017 and 'modelo' in resultados_017:
            resultados_017=resultados_017.rename(columns={'modelo':'variant'})
        nombres_017={'original_24_variables':'XGBoost original','kmeans5_fijo':'XGBoost + K-means fijo',
                     'kmeans5_grilla_30':'XGBoost + K-means, 30 configuraciones',
                     'kmeans5_grilla_36':'XGBoost + K-means, 36 configuraciones'}
        resultados_017['Configuración']=resultados_017['variant'].map(nombres_017).fillna(resultados_017['variant'])
        display(resultados_017[['Configuración','roc_auc','precision_promedio','perdida_logaritmica','brier']].sort_values('roc_auc',ascending=False).round(6))
        tabla=resultados_017.sort_values('roc_auc')
        fig,ax=plt.subplots(figsize=(12,4.5))
        pos=np.arange(len(tabla))
        ax.scatter(tabla['roc_auc'],pos,s=65,color=COLORES[0])
        ax.set_yticks(pos,tabla['Configuración'])
        mn,mx=tabla['roc_auc'].min(),tabla['roc_auc'].max()
        ax.set_xlim(mn-.0004,mx+.0007)
        for j,v in enumerate(tabla['roc_auc']):
            ax.annotate(f'{v:.6f}',(v,j),xytext=(7,0),textcoords='offset points',va='center')
        ax.set(title='XGBoost: predictores originales frente a cinco grupos K-means',
               xlabel='ROC AUC OOF (eje ampliado)',ylabel='')
        cerrar_figura(fig,'13c_xgboost_kmeans5_comparacion')
        elegidos=RAIZ/'experiments'/'experiment017'/'results'/'selected_params_36.csv'
        if elegidos.is_file():display(pd.read_csv(elegidos))
    else:
        print('No se encontraron los agregados del experimento017: consulte su README.')


## 4. Ensambles

Se comparan cuatro estrategias sobre los modelos individuales seleccionados en la sección 3. Todos los ensambles se evalúan sobre los mismos folds externos.

### 4.1 Promedio y bagging

El promedio heterogéneo combina las cuatro familias: $\hat p(x)=\frac14\sum_{j=1}^4\hat p_j(x)$. No requiere nuevos ajustes. Bagging clásico usa cinco muestras bootstrap con reemplazo y el XGBoost seleccionado en cada fold; el preprocesador se ajusta una vez al entrenamiento externo y se mantiene fijo durante el remuestreo. Se promedian probabilidades de las cinco réplicas. No se confunde con el promedio de modelos de validación cruzada que se usa para las predicciones Kaggle.


### 4.2 Stacking con XGBoost

El meta-modelo recibe cuatro columnas de probabilidades, una por familia. Para cada split de búsqueda del meta-modelo, las probabilidades de meta-entrenamiento se construyen con cross-fitting dentro de ese entrenamiento; las de meta-validación provienen de bases ajustadas solo con meta-entrenamiento. Se prueban las **36 configuraciones** del mismo booster. Para reajustar el meta-modelo se generan OOF en todo el entrenamiento externo y se usan las probabilidades externas de las bases de la sección 3.

Los hiperparámetros de las bases se seleccionaron previamente sobre la muestra interna del fold externo y se mantienen fijos en la búsqueda del meta-modelo. Por ello la AUC interna de stacking es condicional a esa selección y no se interpreta como estimación independiente. El fold externo nunca interviene en ninguna selección ni ajuste.


### 4.3 Corrección XGBoost de la mejor regresión logística

Se conserva la representación, regularización y ponderación de la mejor logística de cada fold (elegida por su propia AUC interna). Sobre los predictores originales se aprende un corrector con la misma grilla de 36 configuraciones:

$$
\hat p(x)=\sigma\big(f_{\mathrm{LR}}^*(x)+g_{\mathrm{XGB}}(x)\big).
$$

Los logits se pasan como `base_margin` durante el ajuste y la inferencia. No es stacking de probabilidades ni regresión de residuos explícitos. El ajuste de todos los ensambles se realiza a continuación; sus resultados se conservan en `outputs/experiment021/`.


In [ ]:
# Ensambles: cada predicción externa proviene exclusivamente del entrenamiento externo.
def predicciones_bases(X,y,train_idx,val_idx,elegidos):
    a,b=matrices_por_split(X.iloc[train_idx],[X.iloc[val_idx]])
    columnas=[]
    for familia in FAMILIAS_INDIVIDUALES:
        _,rep,par=elegidos[familia]
        m=nuevo_modelo(familia,par).fit(a[rep],y[train_idx])
        columnas.append(probabilidades(m,b[rep]))
    return np.column_stack(columnas)

def meta_oof(X,y,idx,elegidos,n_splits,seed):
    """OOF de las cuatro bases, entrenadas exclusivamente en índices permitidos."""
    salida=np.full((len(idx),len(FAMILIAS_INDIVIDUALES)),np.nan)
    cv=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    for it,iv in cv.split(idx,y[idx]):
        salida[iv]=predicciones_bases(X,y,idx[it],idx[iv],elegidos)
    assert np.isfinite(salida).all()
    return salida

def cache_stacking(X,y,sub,fold,elegidos):
    # Para cada validación meta, generar OOF solo en meta-train, sin etiquetas meta-val.
    cache=[]
    for j,(it,iv) in enumerate(splits_internos(sub,y,fold)):
        train_idx,val_idx=sub[it],sub[iv]
        mt=meta_oof(X,y,train_idx,elegidos,2,SEMILLA+1000+10*fold+j)
        mv=predicciones_bases(X,y,train_idx,val_idx,elegidos)
        cache.append((mt,mv,y[train_idx],y[val_idx]))
    return cache

def cache_offset(X,y,sub,fold,elegidos):
    _,rep,par=elegidos['Logística']
    cache=[]
    for a,b,yt,yv in cache_interna(X,y,sub,fold):
        lr=nuevo_modelo('Logística',par).fit(a[rep],yt)
        cache.append((a['baseline'],b['baseline'],yt,yv,
                      lr.decision_function(a[rep]),lr.decision_function(b[rep])))
    return cache

def buscar_booster(cache,fold,familia,rep,registros,offset=False):
    opciones=[]
    for par in ParameterGrid(GRILLA_BOOSTER):
        scores=[]
        for paquete in cache:
            a,b,yt,yv=paquete[:4]
            m=nuevo_modelo(familia,par)
            if offset:
                mt,mv=paquete[4:]
                m.fit(a,yt,base_margin=mt)
                pv=m.predict_proba(b,base_margin=mv)[:,1]
            else:
                m.fit(a,yt)
                pv=probabilidades(m,b)
            scores.append(roc_auc_score(yv,pv))
        auc=registro_grilla(registros,fold,familia,rep,par,scores)
        opciones.append((auc,rep,par))
    return max(opciones,key=lambda q:q[0])

def bagging_xgb(a,b,c,y_train,par,fold):
    """Bootstrap de filas con reemplazo; preprocesador fijo ajustado en el outer train."""
    rng=np.random.default_rng(SEMILLA+2000+fold)
    pv=np.zeros(len(b)); pt=np.zeros(len(c))
    for k in range(N_BAGGING):
        boot=rng.integers(0,len(y_train),size=len(y_train))
        while np.unique(y_train[boot]).size!=2:
            boot=rng.integers(0,len(y_train),size=len(y_train))
        m=nuevo_modelo('XGBoost',par).set_params(random_state=SEMILLA+100*fold+k)
        m.fit(a[boot],y_train[boot])
        pv+=probabilidades(m,b)/N_BAGGING
        pt+=probabilidades(m,c)/N_BAGGING
    return pv,pt

def protocolo_ensambles(X,y,X_test,estados):
    y=np.asarray(y,dtype=int)
    resultados=nuevas_predicciones(ENSAMBLES,len(y),len(X_test))
    registros=[]; ganadores=[]
    stack='Stacking XGBoost'; offset='Logística + XGBoost'
    for estado in estados:
        fold,tr,va,sub,elegidos=(estado[k] for k in ('fold','tr','va','sub','elegidos'))
        assert not np.intersect1d(tr,va).size and np.isin(sub,tr).all()
        # 1. Promedio heterogéneo: se reutilizan las probabilidades de la sección 3.
        pv=estado['meta_val'].mean(axis=1); pt=estado['meta_test'].mean(axis=1)
        agregar_prediccion(resultados,'Promedio de familias',va,pv,pt)
        registrar_ganador(ganadores,fold,'Promedio de familias','cuatro familias',{},np.nan,y[va],pv)
        # 2. Stacking: ajustar 36 configuraciones de XGBoost sobre predicciones OOF.
        cache=cache_stacking(X,y,sub,fold,elegidos)
        auc_s,rep_s,par_s=buscar_booster(cache,fold,stack,'cuatro probabilidades OOF',registros)
        del cache
        # Para el meta-modelo final se generan OOF de bases en todo el entrenamiento externo.
        meta_train=meta_oof(X,y,tr,elegidos,3,SEMILLA+3000+fold)
        meta=nuevo_modelo(stack,par_s).fit(meta_train,y[tr])
        pv=probabilidades(meta,estado['meta_val']); pt=probabilidades(meta,estado['meta_test'])
        agregar_prediccion(resultados,stack,va,pv,pt)
        registrar_ganador(ganadores,fold,stack,rep_s,par_s,auc_s,y[va],pv)
        del meta_train,meta
        # 3. Corrección de logits: base logística seleccionada por su propia AUC.
        _,rep_lr,par_lr=elegidos['Logística']
        cache=cache_offset(X,y,sub,fold,elegidos)
        auc_o,rep_o,par_o=buscar_booster(cache,fold,offset,rep_lr,registros,offset=True)
        del cache
        a,b,c=matrices_por_split(X.iloc[tr],[X.iloc[va],X_test])
        lr=nuevo_modelo('Logística',par_lr).fit(a[rep_lr],y[tr])
        booster=nuevo_modelo(offset,par_o).fit(a['baseline'],y[tr],base_margin=lr.decision_function(a[rep_lr]))
        pv=booster.predict_proba(b['baseline'],base_margin=lr.decision_function(b[rep_lr]))[:,1]
        pt=booster.predict_proba(c['baseline'],base_margin=lr.decision_function(c[rep_lr]))[:,1]
        agregar_prediccion(resultados,offset,va,pv,pt)
        registrar_ganador(ganadores,fold,offset,rep_o,par_o,auc_o,y[va],pv)
        del lr,booster
        # 4. Bagging clásico del XGBoost seleccionado: cinco muestras bootstrap completas.
        _,rep_x,par_x=elegidos['XGBoost']
        pv,pt=bagging_xgb(a[rep_x],b[rep_x],c[rep_x],y[tr],par_x,fold)
        agregar_prediccion(resultados,'Bagging XGBoost',va,pv,pt)
        registrar_ganador(ganadores,fold,'Bagging XGBoost',rep_x,{**par_x,'n_bootstrap':N_BAGGING},np.nan,y[va],pv)
        del a,b,c
        print('Fold',fold,'ensambles terminados',flush=True)
    return resultados,pd.DataFrame(registros),pd.DataFrame(ganadores)

if EJECUTAR_PROTOCOLO_COMUN:
    resultados_ensambles,grilla_ensambles,seleccion_ensambles=protocolo_ensambles(entrenamiento[variables],y,prueba[variables],estados_individuales)
    resultados_comunes={**resultados_individuales,**resultados_ensambles}
    resumen_comun=resumir_predicciones(resultados_comunes,y)
    grilla_comun=pd.concat([grilla_individual,grilla_ensambles],ignore_index=True)
    seleccion_comun=pd.concat([seleccion_individual,seleccion_ensambles],ignore_index=True)
    directorio.mkdir(parents=True,exist_ok=True)
    resumen_comun.to_csv(directorio/'summary.csv',index=False)
    grilla_comun.to_csv(directorio/'grid_results.csv',index=False)
    seleccion_comun.to_csv(directorio/'selected_params.csv',index=False)
    np.savez_compressed(directorio/'predictions_PRIVATE.npz',train_id=entrenamiento[IDENTIFICADOR].to_numpy(),
                        test_id=prueba[IDENTIFICADOR].to_numpy(),y=y,
                        **{f'{f}__{k}':v for f,d in resultados_comunes.items() for k,v in d.items()})
elif CARGAR_PROTOCOLO_COMUN:
    resultados_ensambles={f:resultados_comunes[f] for f in ENSAMBLES}
    grilla_ensambles=grilla_comun[grilla_comun.familia.isin(ENSAMBLES)]
    seleccion_ensambles=seleccion_comun[seleccion_comun.familia.isin(ENSAMBLES)]
if resultados_comunes:
    mostrar_grillas(grilla_ensambles,seleccion_ensambles,('Stacking XGBoost','Logística + XGBoost'),'grilla_ensambles')
    display(resumir_predicciones(resultados_ensambles,y).round(6))
    display(seleccion_ensambles.round(6))
else:
    print('Modo histórico: los ensambles nuevos no se han ejecutado.')


### 4.4 Referencia histórica (solo modo histórico)

Resultados anteriores de la logística y el corrector, sin mezclarlos con los ocho candidatos nuevos.

In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    ruta_lr = RAIZ / 'experiments' / 'experiment014' / 'results' / 'summary.csv'
    ruta_ic_lr = RAIZ / 'experiments' / 'experiment014' / 'results' / 'paired_auc_bootstrap.csv'
    if ruta_lr.is_file():
        lr_tabla = pd.read_csv(ruta_lr)
        nombres_lr = {'baseline':'Variables originales', 'kmeans':'Con K-means',
                      'dbscan':'Con DBSCAN','combined':'K-means + DBSCAN'}
        lr_tabla['Representación'] = lr_tabla['variant'].map(nombres_lr)
        lr_tabla = lr_tabla.sort_values('roc_auc', ascending=False)
        display(lr_tabla[['Representación','roc_auc','average_precision','log_loss','brier']].round(6))
        mejor_lr_aparente = str(lr_tabla.iloc[0]['variant'])
        REPRESENTACION_LOGISTICA_REFERENCIA = 'baseline'
        print('Mayor AUC observada en la etapa logística:', nombres_lr[mejor_lr_aparente])
        if ruta_ic_lr.is_file() and mejor_lr_aparente != 'baseline':
            intervalos_lr = pd.read_csv(ruta_ic_lr)
            fila = intervalos_lr.loc[intervalos_lr['variant'] == mejor_lr_aparente].iloc[0]
            display(pd.DataFrame([{'Comparación':f'{nombres_lr[mejor_lr_aparente]} frente a originales',
                                   'Diferencia AUC':fila['delta_auc'],
                                   'Límite inferior 95 %':fila['ci95_low'],
                                   'Límite superior 95 %':fila['ci95_high']}]).round(6))
            if fila['ci95_low'] <= 0 <= fila['ci95_high']:
                print('Decisión: conservar representación original por parsimonia; no se demostró superioridad.')
            else:
                print('La señal requiere evaluación anidada o independiente antes de seleccionarla.')
        fig, ax = plt.subplots(figsize=(10.5,4.8))
        tabla_plot = lr_tabla.sort_values('roc_auc')
        ax.scatter(tabla_plot['roc_auc'],np.arange(len(tabla_plot)),s=75,color=COLORES[0])
        ax.set_yticks(np.arange(len(tabla_plot)),tabla_plot['Representación'])
        ax.set_xlim(float(lr_tabla.roc_auc.min())-.00015,float(lr_tabla.roc_auc.max())+.00025)
        for j,v in enumerate(tabla_plot['roc_auc']):
            ax.annotate(f'{v:.6f}', (v,j), xytext=(7,0), textcoords='offset points', va='center',fontsize=9)
        ax.set(title='Regresión logística: comparación de representaciones',
               xlabel='ROC AUC fuera de partición (eje ampliado)',ylabel='')
        cerrar_figura(fig,'19_seleccion_base_logistica')
    else:
        REPRESENTACION_LOGISTICA_REFERENCIA = 'baseline'
        print('Tabla del experimento014 no disponible. Se conserva la base original sin inventar resultados.')



In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    if EJECUTAR_BUSQUEDA_HIBRIDA:
        import subprocess
        script=RAIZ/'experiments'/'experiment018'/'tuning_offset.py'
        if not script.is_file():raise FileNotFoundError('Se requiere el script privado de experiment018 incluido en el ZIP de reproducción.')
        comando=[sys.executable,str(script),'--data',str(RAIZ/'playground-series-s6e11.zip'),
                 '--output',str(RAIZ/'outputs'/'experiment018'/'full_699960'),
                 '--inner-rows','36000','--outer-folds','5','--inner-folds','3',
                 '--n-estimators','240','--jobs','4']
        subprocess.run(comando,check=True)

    RUTA_HIBRIDO=RAIZ/'experiments'/'experiment018'/'resultados'
    if not RUTA_HIBRIDO.exists():RUTA_HIBRIDO=RAIZ/'outputs'/'experiment018'/'full_699960'
    archivo_hibrido=RUTA_HIBRIDO/'summary.csv'
    if archivo_hibrido.is_file():
        res_hibrido=pd.read_csv(archivo_hibrido).sort_values('roc_auc',ascending=False)
        mapa={'baseline':'Datos originales','kmeans':'Logística con K-means','dbscan':'Logística con DBSCAN','combined':'Logística con ambos'}
        res_hibrido['Representación']=res_hibrido['variant'].map(mapa)
        display(res_hibrido[['Representación','roc_auc','average_precision','log_loss','brier']].round(6))
        fig,ax=plt.subplots(figsize=(11,5))
        sns.scatterplot(data=res_hibrido,x='roc_auc',y='Representación',s=130,color=COLORES[0],ax=ax)
        for j,v in enumerate(res_hibrido.roc_auc):ax.annotate(f'{v:.6f}',(v,j),xytext=(6,0),textcoords='offset points')
        ax.set(title='Corrección XGBoost: búsqueda de representación e hiperparámetros',xlabel='ROC AUC OOF (eje ampliado)',ylabel='')
        cerrar_figura(fig,'13e_busqueda_hibrida_con_clustering')
        archivo_parametros=RUTA_HIBRIDO/'best_inner_params.csv'
        if archivo_parametros.is_file():
            display(pd.read_csv(archivo_parametros).round(6))
    else:
        print('No se encontraron resultados completos de la búsqueda anidada; no se inventan AUC.')


## 5. Selección final y evaluación

### 5.1 Comparación de los ocho candidatos

Se comparan las cuatro familias individuales y los cuatro ensambles mediante ROC AUC OOF, precisión promedio, log loss y Brier. Se selecciona el máximo observado de ROC AUC; la evaluación posterior utiliza sus predicciones y no el ganador histórico por defecto.


In [ ]:
if resultados_comunes:
    display(resumen_comun.round(6))
    tabla=resumen_comun.sort_values('roc_auc')
    fig,ax=plt.subplots(figsize=(12,5.5))
    sns.scatterplot(data=tabla,x='roc_auc',y='familia',hue=tabla.familia.isin(ENSAMBLES).map({True:'Ensamble',False:'Individual'}),s=95,ax=ax)
    for j,v in enumerate(tabla.roc_auc):
        ax.annotate(f'{v:.6f}',(v,j),xytext=(7,0),textcoords='offset points',va='center')
    ax.set(title='Comparación final: individuales y ensambles',xlabel='ROC AUC OOF',ylabel='')
    ax.legend(title='Tipo')
    cerrar_figura(fig,'comparacion_final_8_candidatos')
    print('Finalista del protocolo nuevo:',resumen_comun.iloc[0].familia)
else:
    print('Modo histórico: consultar la comparación de referencia a continuación.')


### 5.2 Referencia histórica e incertidumbre

Los ensambles históricos, tiempos y bootstrap se muestran solo en modo histórico. No hay intervalos nuevos hasta analizar los OOF nuevos.

In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    # Los resultados agregados se conservan con su origen y nunca se confunden con nuevas ejecuciones.
    p=DATOS_AGREGADOS/'five_way_comparison.csv'
    if not p.exists(): raise FileNotFoundError(f'No se encontró {p}; conserve el directorio resultados incluido con el notebook.')
    comparacion=pd.read_csv(p)
    print('Columnas originales:',list(comparacion.columns))
    display(comparacion.head())


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    def media_de_probabilidades(*predicciones):
        matriz=np.column_stack([np.asarray(p,dtype=float) for p in predicciones])
        if not (np.isfinite(matriz).all() and np.min(matriz)>=0 and np.max(matriz)<=1):
            raise ValueError('Los ensambles deben recibir probabilidades válidas.')
        return matriz.mean(axis=1)

    def entrenar_correccion_logits(X_ent,y_ent,X_val,regresion_logistica,arboles=240,dispositivo='cpu'):
        """Modelo secuencial original: corregir logits con pérdida logística."""
        from xgboost import XGBClassifier
        regresion_logistica.fit(X_ent,y_ent)
        logit_ent=np.asarray(regresion_logistica.decision_function(X_ent),dtype=np.float32)
        booster=XGBClassifier(n_estimators=arboles,max_depth=3,learning_rate=.05,
                              min_child_weight=5,subsample=.85,colsample_bytree=.85,
                              tree_method='hist',device=dispositivo,n_jobs=-1,
                              objective='binary:logistic',eval_metric='logloss',random_state=SEMILLA)
        booster.fit(X_ent,y_ent,base_margin=logit_ent)
        logit_val=np.asarray(regresion_logistica.decision_function(X_val),dtype=np.float32)
        return booster.predict_proba(X_val,base_margin=logit_val)[:,1]

    def construir_objetivo_residual(y_ent,p_logistica_ent):
        """Objetivo explícito: residuos de probabilidades, no logits."""
        return np.asarray(y_ent,dtype=float)-np.asarray(p_logistica_ent,dtype=float)

    def corregir_probabilidades(p_logistica,p_residuo_estimada):
        """Se recorta la suma para mantenerla dentro de [0,1]."""
        return np.clip(np.asarray(p_logistica)+np.asarray(p_residuo_estimada),0,1)

    def matriz_para_stacking(diccionario_oof,nombres):
        """El bosque de nivel 2 solo usa una columna de probabilidades OOF por base."""
        return np.column_stack([np.asarray(diccionario_oof[n],dtype=float) for n in nombres])

    print('Definiciones disponibles. No se reentrenaron ensambles en esta sección.')


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    traduccion_estrategias={
     'best_single':'XGBoost individual',
     'ensemble1_blend':'Promedio de probabilidades',
     'ensemble2_logit_offset':'Regresión logística corregida con XGBoost',
     'ensemble3_explicit_residual':'Corrección explícita de residuos',
     'ensemble4_rf_stacking':'Stacking con bosque aleatorio'}
    col_modelo='strategy' if 'strategy' in comparacion else 'name'
    comparacion['Estrategia']=comparacion[col_modelo].map(traduccion_estrategias).fillna(comparacion[col_modelo])
    metricas=['roc_auc','average_precision','log_loss','brier']
    for metrica in metricas: assert metrica in comparacion
    orden=comparacion.sort_values('roc_auc',ascending=False)
    display(orden[['Estrategia']+metricas].round(6))
    tabla=orden.iloc[::-1]
    fig,ax=plt.subplots(figsize=(12.8,5.1))
    sns.barplot(data=tabla,x='roc_auc',y='Estrategia',color=COLORES[0],ax=ax)
    valores=tabla['roc_auc'].to_numpy(dtype=float)
    minimo=float(valores.min());maximo=float(valores.max())
    ax.set_xlim(minimo-.0009,maximo+.0014)
    for yidx,v in enumerate(valores):
        ax.annotate(f'{v:.6f}',xy=(v,yidx),xytext=(6,0),textcoords='offset points',va='center',fontsize=9)
    ax.axvline(float(orden.loc[orden[col_modelo]=='best_single','roc_auc'].iloc[0]),
               ls='--',color=COLORES[3],lw=1.25,label='Mejor modelo individual')
    ax.set(title='Mejor modelo individual frente a cuatro estrategias de ensamble',
           xlabel='ROC AUC fuera de partición (escala ampliada)',ylabel='')
    ax.legend(loc='lower right',fontsize=9)
    cerrar_figura(fig,'14_comparacion_cinco_estrategias')


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    ruta_tiempos=RAIZ/'experiments'/'experiment018'/'resultados'/'tiempos_historicos.csv'
    if ruta_tiempos.exists():
        eficiencia=pd.read_csv(ruta_tiempos).sort_values('roc_auc',ascending=False)
        fig,ax=plt.subplots(figsize=(12.5,6))
        sns.scatterplot(data=eficiencia,x='segundos_booster_por_fold',y='roc_auc',
                        hue='familia',hue_order=eficiencia.familia.drop_duplicates().tolist(),palette='dark',s=130,ax=ax)
        for r in eficiencia.itertuples():
            ax.annotate(r.nombre,(r.segundos_booster_por_fold,r.roc_auc),xytext=(5,6),textcoords='offset points',fontsize=8)
        ax.set(title='ROC AUC frente a tiempo del ajuste supervisado',xlabel='Segundos por fold: booster (no incluye preparación ni logística)',ylabel='ROC AUC OOF')
        ax.legend(title='Familia')
        cerrar_figura(fig,'20_roc_auc_vs_tiempo')
        display(eficiencia.round(5))


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    bootstrap=pd.read_csv(DATOS_AGREGADOS/'bootstrap_experiment010.csv')
    bootstrap=bootstrap.sort_values('diferencia_auc')
    fig,ax=plt.subplots(figsize=(12,6.2))
    for j,fila in enumerate(bootstrap.itertuples()):
        ax.plot([fila.ic95_inferior,fila.ic95_superior],[j,j],lw=3,color=COLORES[0])
        ax.scatter(fila.diferencia_auc,j,s=60,color=COLORES[3],zorder=3)
    ax.axvline(0,linestyle='--',color='black',label='Sin diferencia de ROC AUC')
    ax.set(yticks=np.arange(len(bootstrap)),yticklabels=bootstrap['estrategia'].tolist(),
           xlabel='Diferencia respecto de XGBoost individual (ROC AUC)',ylabel='',
           title='Intervalos percentiles del bootstrap emparejado (95 %)')
    ax.legend()
    cerrar_figura(fig,'17_bootstrap_comparacion_ensambles')
    display(bootstrap.sort_values('roc_auc_oof',ascending=False).round(6))


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    ruta_boot018=RAIZ/'experiments'/'experiment018'/'resultados'/'paired_auc_bootstrap.csv'
    if ruta_boot018.exists():
        boot018=pd.read_csv(ruta_boot018).sort_values('delta_auc')
        nombres018={'kmeans':'K-means en logística','dbscan':'DBSCAN en logística','combined':'K-means y DBSCAN en logística'}
        boot018['Representación']=boot018['variant'].map(nombres018)
        fig,ax=plt.subplots(figsize=(11.8,5.1))
        for j,fila in enumerate(boot018.itertuples()):
            ax.plot([fila.ci95_low,fila.ci95_high],[j,j],color=COLORES[0],lw=3,alpha=.65)
            ax.scatter(fila.delta_auc,j,color=COLORES[3],s=60,zorder=3)
        ax.axvline(0,linestyle='--',color='black',label='Sin diferencia')
        ax.set(yticks=np.arange(len(boot018)),yticklabels=boot018['Representación'],
               xlabel='Diferencia de ROC AUC OOF frente a la logística sin clusters',
               ylabel='',title='Bootstrap emparejado de la búsqueda híbrida (IC 95 %)')
        ax.legend()
        cerrar_figura(fig,'19_bootstrap_representaciones_hibrido')
        display(boot018[['Representación','delta_auc','ci95_low','ci95_high']].round(6))
    else:
        print('Falta el resumen de bootstrap del experimento018; no se crean intervalos artificiales.')


In [ ]:
if not (EJECUTAR_PROTOCOLO_COMUN or CARGAR_PROTOCOLO_COMUN):
    ruta_nested=RAIZ/'experiments'/'experiment018'/'resultados'
    res_nested=ruta_nested/'joint_selected_summary.csv'
    selecciones=ruta_nested/'joint_selected_fold_choices.csv'
    bootstrap_nested=ruta_nested/'joint_selected_bootstrap.csv'
    if all(p.is_file() for p in (res_nested,selecciones,bootstrap_nested)):
        nested=pd.read_csv(res_nested)
        opciones=pd.read_csv(selecciones)
        bn=pd.read_csv(bootstrap_nested)
        display(opciones[['fold','chosen_representation','inner_auc','max_depth','learning_rate']].round(6))
        display(nested.round(6))
        display(bn[['delta_auc','ci95_low','ci95_high']].round(6))
        fig,ejes=plt.subplots(1,2,figsize=(13.5,4.7))
        pal={'baseline':COLORES[0],'kmeans':COLORES[2]}
        sns.scatterplot(data=opciones,x='fold',y='inner_auc',hue='chosen_representation',
                        hue_order=['baseline','kmeans'],palette=pal,s=140,ax=ejes[0])
        bajo,alto=opciones['inner_auc'].agg(['min','max'])
        ejes[0].set(ylim=(bajo-.002,alto+.002),xticks=[1,2,3,4,5],
                    title='Representación elegida por validación interna',
                    xlabel='Pliegue externo',ylabel='ROC AUC interna (eje ampliado)')
        for fila in opciones.itertuples():
            ejes[0].annotate(f'{fila.inner_auc:.4f}',(fila.fold,fila.inner_auc),
                             xytext=(0,11),textcoords='offset points',ha='center',fontsize=8)
        handles,labels=ejes[0].get_legend_handles_labels()
        ejes[0].legend(handles,['Variables originales' if x=='baseline' else 'K-means' for x in labels],
                       title='Datos seleccionados',loc='lower left')
        fila=bn.iloc[0];escala=10000
        ejes[1].plot([fila.ci95_low*escala,fila.ci95_high*escala],[0,0],color=COLORES[0],lw=4)
        ejes[1].scatter(fila.delta_auc*escala,0,color=COLORES[3],s=90,zorder=4)
        ejes[1].axvline(0,color='black',linestyle='--')
        ejes[1].set(yticks=[0],yticklabels=['Selector conjunto'],
                    title='Diferencia frente a variables originales (IC 95 %)',
                    xlabel='Diferencia de ROC AUC × 10⁴',ylabel='')
        margen=max((fila.ci95_high-fila.ci95_low)*escala*.15,.15)
        ejes[1].set_xlim(fila.ci95_low*escala-margen,fila.ci95_high*escala+margen)
        cerrar_figura(fig,'26_selector_anidado_representaciones')
    else:
        print('El selector anidado requiere los resultados agregados de experiment018.')


### 5.3 Elección del finalista

En el protocolo nuevo se selecciona el máximo ROC AUC OOF entre los ocho candidatos. Las curvas y el CSV de Kaggle se construyen con sus probabilidades. En modo histórico únicamente se conserva el ganador previamente documentado (offset, AUC 0.946205); no se presenta como resultado de experiment021. La selección por máximo OOF puede introducir optimismo y no prueba superioridad estadística.


In [ ]:
def localizar_npz():
    ubicaciones=[RAIZ/'outputs'/'ensemble_extension'/'five_way_oof_and_test_predictions.npz',
                 RAIZ.parent/'outputs'/'ensemble_extension'/'five_way_oof_and_test_predictions.npz',
                 RAIZ/'ensemble_extension'/'five_way_oof_and_test_predictions.npz']
    for u in ubicaciones:
        if u.is_file(): return ('archivo',u)
    for u in [RAIZ/'ensemble_extension.zip',RAIZ.parent/'ensemble_extension.zip']:
        if u.is_file():
            with zipfile.ZipFile(u) as z:
                claves=[a for a in z.namelist() if a.endswith('five_way_oof_and_test_predictions.npz')]
                if claves:return ('zip',(u,claves[0]))
    return None
fuente_oof=None if resultados_comunes else localizar_npz()
resultados_oof={}
if fuente_oof:
    tipo,ref=fuente_oof
    if tipo=='archivo':
        almacenamiento=np.load(ref,allow_pickle=False)
    else:
        ruta,nombre=ref
        with zipfile.ZipFile(ruta) as z:
            almacenamiento=np.load(io.BytesIO(z.read(nombre)),allow_pickle=False)
    try:
        for nombre in traduccion_estrategias:
            clave_oof=f'{nombre}__oof';clave_prueba=f'{nombre}__test'
            if clave_oof not in almacenamiento or clave_prueba not in almacenamiento:continue
            po=np.asarray(almacenamiento[clave_oof],dtype=np.float64)
            pt=np.asarray(almacenamiento[clave_prueba],dtype=np.float64)
            if po.shape!=(len(y),) or pt.shape!=(len(prueba),):
                raise ValueError(f'Longitud incorrecta de {nombre}: puede ser un orden o conjunto distintos')
            if not(np.isfinite(po).all() and np.isfinite(pt).all() and 0<=po.min()<=po.max()<=1 and 0<=pt.min()<=pt.max()<=1):
                raise ValueError('Probabilidades no válidas o no finitas: '+nombre)
            resultados_oof[nombre]={'oof':po,'test':pt}
    finally:
        almacenamiento.close()
    for fila in comparacion.itertuples():
        codigo=getattr(fila,col_modelo)
        if codigo in resultados_oof:
            observado=roc_auc_score(y,resultados_oof[codigo]['oof'])
            if abs(observado-float(fila.roc_auc))>2e-5:
                raise AssertionError(f'Las predicciones de {codigo} no concuerdan con el resumen agregado.')
    print('Vectores OOF privados verificados:',len(resultados_oof))
elif not resultados_comunes:
    print('No se localizaron las predicciones OOF privadas; los resultados agregados permanecen disponibles.')
if resultados_comunes:
    resultados_oof=resultados_comunes
    traduccion_estrategias={f:f for f in resultados_comunes}
    print('Evaluación final: predicciones del nuevo protocolo común.')


In [ ]:
seleccionada=str(resumen_comun.iloc[0]['familia']) if resultados_comunes else 'ensemble2_logit_offset'
print('Modelo seleccionado:',seleccionada)
if seleccionada in resultados_oof:
    prob=np.asarray(resultados_oof[seleccionada]['test'],dtype=float)
    if prob.shape!=(len(prueba),) or not np.isfinite(prob).all() or np.min(prob)<0 or np.max(prob)>1:
        raise ValueError('El vector de probabilidades para Kaggle no es válido')
    envio=plantilla_envio[[IDENTIFICADOR]].copy()
    envio[OBJETIVO]=prob
    assert envio[IDENTIFICADOR].equals(prueba[IDENTIFICADOR])
    ruta=SALIDA/'envio_final_kaggle.csv'
    envio.to_csv(ruta,index=False)
    display(envio.head(6))
    print('Archivo de Kaggle creado:',ruta,'| filas:',len(envio))
else:
    print('No se creó un envío: es necesario recuperar las predicciones de prueba del modelo seleccionado.')

### 5.4 Curvas ROC, precisión–exhaustividad y calibración

Estas curvas se recalculan con las etiquetas reales y las probabilidades **guardadas** de cada estrategia. Por seguridad, los vectores OOF individuales no están en este paquete público. Si no están presentes en su equipo, se omiten estas gráficas y se conservan las métricas verificadas sin interpolarlas artificialmente.


In [ ]:
if resultados_oof:
    resultados_oof=dict(sorted(resultados_oof.items(),key=lambda t:roc_auc_score(y,t[1]['oof']),reverse=True))
    orden_pr=dict(sorted(resultados_oof.items(),key=lambda t:average_precision_score(y,t[1]['oof']),reverse=True))
    fig,ejes=plt.subplots(1,2,figsize=(14.5,5.5))
    for i,(nombre,datos) in enumerate(resultados_oof.items()):
        prob=datos['oof']; color=COLORES[i%len(COLORES)]
        fpr,tpr,_=roc_curve(y,prob)
        sel=np.unique(np.linspace(0,len(fpr)-1,min(2500,len(fpr)),dtype=int))
        fpr,tpr=fpr[sel],tpr[sel]
        prec,rec,_=precision_recall_curve(y,prob,drop_intermediate=True)
        sel=np.unique(np.linspace(0,len(prec)-1,min(2500,len(prec)),dtype=int))
        prec,rec=prec[sel],rec[sel]
        ejes[0].plot(fpr,tpr,color=color,alpha=1 if i==0 else .5,label=f'{traduccion_estrategias[nombre]} · {roc_auc_score(y,prob):.4f}')

    for i,(nombre,datos) in enumerate(orden_pr.items()):
        prec,rec,_=precision_recall_curve(y,datos['oof'],drop_intermediate=True)
        sel=np.unique(np.linspace(0,len(prec)-1,min(2500,len(prec)),dtype=int))
        color=COLORES[list(resultados_oof).index(nombre)%len(COLORES)]
        ejes[1].plot(rec[sel],prec[sel],color=color,alpha=1 if i==0 else .5,
                     label=f'{traduccion_estrategias[nombre]} · {average_precision_score(y,datos["oof"]):.4f}')
    ejes[0].plot([0,1],[0,1],ls='--',color='gray',label='Azar')
    ejes[0].set(title='Curvas ROC en predicciones OOF',xlabel='Tasa de falsos positivos',ylabel='Tasa de verdaderos positivos')
    ejes[1].axhline(np.mean(y),ls='--',color='gray',label='Prevalencia de fraude')
    ejes[1].set(title='Curvas precisión–exhaustividad',xlabel='Exhaustividad (sensibilidad)',ylabel='Precisión')
    for ax in ejes:ax.legend(fontsize=8,loc='best')
    cerrar_figura(fig,'15_curvas_roc_y_precision_exhaustividad')
    fig,ax=plt.subplots(figsize=(9,6))
    for i,(nombre,datos) in enumerate(resultados_oof.items()):
        tabla=pd.DataFrame({'real':y,'riesgo':datos['oof']})
        tabla['intervalo']=pd.qcut(tabla['riesgo'],q=15,duplicates='drop')
        grupo=tabla.groupby('intervalo',observed=True).agg(p_pred=('riesgo','mean'),p_real=('real','mean'))
        ax.plot(grupo['p_pred'],grupo['p_real'],marker='o',markersize=3,alpha=1 if i==0 else .5,label=traduccion_estrategias[nombre],color=COLORES[i%len(COLORES)])
    ax.plot([0,1],[0,1],color='black',ls='--',label='Calibración ideal')
    ax.set(title='Calibración observada por intervalos de riesgo OOF',xlabel='Probabilidad media estimada',ylabel='Frecuencia observada de fraude',xlim=(0,1),ylim=(0,1))
    ax.legend(fontsize=8)
    cerrar_figura(fig,'16_calibracion_oof')
else:
    print('Curvas y calibración omitidas por ausencia de vectores OOF privados.')

### 5.5 Determinación del umbral de alerta

Las probabilidades pueden convertirse en decisiones mediante `predicción ≥ umbral`. Reducir falsos negativos suele incrementar falsos positivos. Se estudia el umbral convencional 0.50, el que maximiza F1 y el **mayor umbral compatible con 98 % de exhaustividad** sobre datos de selección, solo como objetivo ilustrativo. Se reportan VP, VN, FP, FN, sus porcentajes, F1, precisión, recall, exactitud y ROC AUC.

Los umbrales de evaluación cruzada se eligen en cuatro folds y se aplican al quinto; esto reduce optimismo pero no equivale a validación independiente ni resuelve dependencias por cliente.


In [ ]:
from sklearn.metrics import (confusion_matrix,f1_score,accuracy_score,precision_score,recall_score,
                             balanced_accuracy_score)
OBJETIVO_RECALL=0.98

def elegir_umbral(y_real,p,objetivo=OBJETIVO_RECALL):
    positivos=np.sort(np.asarray(p)[np.asarray(y_real)==1])
    if not len(positivos):raise ValueError('No hay observaciones fraudulentas')
    return float(positivos[max(0,int(np.floor((1-objetivo)*len(positivos))))])

def metricas_alerta(y_real,p,umbral,nombre,politica):
    b=np.asarray(p)>=umbral
    vn,fp,fn,vp=confusion_matrix(y_real,b,labels=[0,1]).ravel()
    return {'Estrategia':nombre,'Política':politica,'Umbral':umbral,
            'VP':int(vp),'FN':int(fn),'FP':int(fp),'VN':int(vn),
            'Recall':recall_score(y_real,b),'Precisión':precision_score(y_real,b,zero_division=0),
            'F1':f1_score(y_real,b,zero_division=0),'Exactitud':accuracy_score(y_real,b),
            'ROC AUC':roc_auc_score(y_real,p),'Tasa de falsas alarmas':fp/(fp+vn)}

predicciones={k:v['oof'] for k,v in resultados_oof.items()}
# Los resultados nuevos se incorporan si el propietario conserva el archivo OOF local.
archivo=RAIZ/'outputs'/'experiment017'/'full_699960'/'checkpoint_PRIVATE.npz'
if not resultados_comunes and archivo.is_file():
    with np.load(archivo,allow_pickle=False) as z:
        for key,field in [('xgb_kmeans5_fijo','aug_fixed_oof'),('xgb_kmeans5_grilla30','aug_grid_oof')]:
            p=np.asarray(z[field])
            if len(p)==len(y) and np.isfinite(p).all():predicciones[key]=p
archivo30=RAIZ/'outputs'/'experiment017'/'depth30'/'checkpoint_depth30_PRIVATE.npz'
if not resultados_comunes and archivo30.is_file():
    with np.load(archivo30,allow_pickle=False) as z:
        p=np.asarray(z['grid_depth30_oof'])
        if len(p)==len(y) and np.isfinite(p).all():predicciones['xgb_kmeans5_grilla36']=p
if predicciones:
    nombres={**traduccion_estrategias,'xgb_kmeans5_fijo':'XGBoost + K-means fijo',
             'xgb_kmeans5_grilla30':'XGBoost + K-means 30',
             'xgb_kmeans5_grilla36':'XGBoost + K-means 36'}
    registros=[];cruzados=[];predicciones_cruzadas={}
    pliegues=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEMILLA).split(np.zeros(len(y)),y))
    for codigo,p in predicciones.items():
        nombre=nombres.get(codigo,codigo)
        t=elegir_umbral(y,p)
        registros.append(metricas_alerta(y,p,.5,nombre,'Umbral 0,50'))
        pp,rr,tt=precision_recall_curve(y,p)
        f1=2*pp[:-1]*rr[:-1]/np.maximum(pp[:-1]+rr[:-1],1e-12)
        registros.append(metricas_alerta(y,p,float(tt[np.argmax(f1)]),nombre,'Máximo F1 OOF'))
        registros.append(metricas_alerta(y,p,t,nombre,'Recall 98 % OOF'))
        decisiones=np.zeros(len(y),dtype=bool)
        for tr,va in pliegues:
            decisiones[va]=p[va]>=elegir_umbral(y[tr],p[tr])
        predicciones_cruzadas[codigo]=decisiones
        vn,fp,fn,vp=confusion_matrix(y,decisiones,labels=[0,1]).ravel()
        cruzados.append({'Estrategia':nombre,'FN':fn,'FP':fp,
                         'Recall cruzado':vp/(vp+fn),'F1 cruzado':f1_score(y,decisiones),
                         'Exactitud cruzada':accuracy_score(y,decisiones),
                         'Falsas alarmas':fp/(vn+fp),'ROC AUC OOF':roc_auc_score(y,p),
                         'Umbral OOF':t,'Código':codigo})
    tabla_alerta=pd.DataFrame(registros)
    tabla_cruzada=pd.DataFrame(cruzados).sort_values(['Falsas alarmas','FN'])
    display(tabla_alerta[['Estrategia','Política','Umbral','VP','FN','FP','VN','Recall','Precisión','F1','Exactitud','ROC AUC']].round(6))
    display(tabla_cruzada.round(6))
    tabla_alerta.to_csv(SALIDA/'metricas_alerta.csv',index=False)
    tabla_cruzada.to_csv(SALIDA/'metricas_alerta_cruzadas.csv',index=False)
    # Matriz del modelo seleccionado por ROC AUC; las alternativas operativas se mantienen en la tabla.
    elegido=tabla_cruzada.loc[tabla_cruzada['Código']==seleccionada].iloc[0] if seleccionada in predicciones else tabla_cruzada.iloc[0]
    matriz=confusion_matrix(y,predicciones_cruzadas[elegido['Código']],labels=[0,1])
    fig,ax=plt.subplots(figsize=(8,6))
    sns.heatmap(matriz,annot=anotaciones_matriz(matriz),fmt='',cmap='Blues',cbar=False,
       xticklabels=['Legítima predicha','Alerta de fraude'],
       yticklabels=['Legítima real','Fraude real'],ax=ax)
    ax.set(title='Matriz de confusión cruzada · objetivo de exhaustividad 98 %',
           xlabel='Predicción',ylabel='Clase real')
    cerrar_figura(fig,'21_matriz_confusion_umbral')
    fig,ejes=plt.subplots(1,2,figsize=(14,6))
    sns.barplot(data=tabla_cruzada,y='Estrategia',x='FN',color=COLORES[1],ax=ejes[0])
    sns.barplot(data=tabla_cruzada,y='Estrategia',x='Falsas alarmas',color=COLORES[0],ax=ejes[1])
    ejes[0].set(title='Fraudes no detectados',xlabel='Falsos negativos',ylabel='')
    ejes[1].set(title='Tasa de alertas falsas',xlabel='Falsas alarmas / transacciones legítimas',ylabel='')
    cerrar_figura(fig,'22_compromiso_errores_operativos')
else:
    archivo=RAIZ/'experiments'/'experiment017'/'results'/'alert_metrics_crossfold.csv'
    if archivo.is_file():
        display(pd.read_csv(archivo)[['modelo','FN_cv','FP_cv','recall_cv','f1_cv','accuracy_cv','roc_auc_oof']].round(6))
        print('Resumen agregado disponible; el gráfico de confusión requiere vectores OOF locales.')
    else:
        print('Sin predicciones OOF: no se inventan métricas binarias.')

### 5.6 Matriz de confusión, F1 y errores operativos

La matriz de confusión informa conteos y porcentajes sobre el conjunto OOF. Estas figuras comparan políticas de decisión y la sensibilidad del F1 ante el umbral; un objetivo de recall del 98 % puede producir un volumen muy elevado de alertas falsas. Los resultados usan predicciones OOF reales, no probabilidades inventadas.


In [ ]:
if predicciones:
    modelo=elegido['Estrategia']
    p_operativo=np.asarray(predicciones[elegido['Código']],dtype=float)
    politicas_modelo=tabla_alerta[tabla_alerta['Estrategia']==modelo]
    fig,ejes=plt.subplots(2,2,figsize=(13.5,9))
    for ax,(col,titulo) in zip(ejes.ravel(),[('F1','F1'),('Precisión','Precisión'),
                                           ('Recall','Exhaustividad'),('Exactitud','Exactitud')]):
        tab=politicas_modelo.sort_values(col,ascending=False)
        sns.barplot(data=tab,x=col,y='Política',color=COLORES[0],ax=ax)
        ax.set(title=titulo+' según la política',xlabel=titulo,ylabel='',xlim=(0,1.02))
    cerrar_figura(fig,'23_metricas_f1_precision_recall_accuracy')
    pp,rr,tt=precision_recall_curve(y,p_operativo,drop_intermediate=True)
    f1=2*pp[:-1]*rr[:-1]/np.maximum(pp[:-1]+rr[:-1],1e-12)
    selec=np.unique(np.linspace(0,len(tt)-1,min(2500,len(tt)),dtype=int))
    fig,ax=plt.subplots(figsize=(12,5.7))
    ax.plot(np.maximum(tt[selec],1e-6),rr[:-1][selec],color=COLORES[1],alpha=.5,label='Exhaustividad')
    ax.plot(np.maximum(tt[selec],1e-6),pp[:-1][selec],color=COLORES[2],alpha=.5,label='Precisión')
    ax.plot(np.maximum(tt[selec],1e-6),f1[selec],color=COLORES[0],alpha=1,lw=2.4,label='F1')
    ax.axvline(tt[np.argmax(f1)],color=COLORES[3],ls='--',label='Máximo F1')
    ax.axvline(float(elegido['Umbral OOF']),color=COLORES[4],ls='--',label='Objetivo 98 %')
    ax.set(xscale='log',title='F1, precisión y exhaustividad según el umbral',
           xlabel='Umbral (escala logarítmica)',ylabel='Métrica (0 a 1)',ylim=(0,1.03))
    ax.legend()
    cerrar_figura(fig,'24_f1_precision_recall_vs_umbral')
    fig,ejes=plt.subplots(1,2,figsize=(15,6))
    f1_ranking=tabla_cruzada.sort_values('F1 cruzado',ascending=False)
    sns.barplot(data=f1_ranking,x='F1 cruzado',y='Estrategia',color=COLORES[0],ax=ejes[0])
    ejes[0].set(title='F1 con umbral seleccionado en otros folds',xlabel='F1 cruzado',ylabel='')
    alertas_ranking=tabla_cruzada.sort_values('FP',ascending=True)
    sns.barplot(data=alertas_ranking,x='FP',y='Estrategia',color=COLORES[1],ax=ejes[1])
    ejes[1].set(title='Alertas sobre operaciones legítimas',xlabel='Falsos positivos',ylabel='')
    cerrar_figura(fig,'25_f1_y_alertas_por_modelo')

else:
    print('Se requieren predicciones OOF privadas para las curvas de umbral y F1.')

## 6. Conclusiones y limitaciones

El informe contrasta representaciones, cuatro clasificadores individuales y cuatro ensambles. La grilla XGBoost de 36 configuraciones es idéntica para el clasificador individual, el meta-modelo y el corrector de logits. El bagging emplea remuestreo con reemplazo y el promedio de familias reutiliza predicciones sin ajuste adicional.

Los resultados nuevos solo se interpretan tras ejecutar el protocolo; no se mezclan con los históricos. La selección de base y meta-modelo reutiliza datos internos, de modo que las métricas externas OOF son la referencia comparativa, aunque el máximo entre candidatos conserva optimismo post-selección. Los umbrales son ilustrativos y Kaggle recibe probabilidades.


In [ ]:
catalogo=pd.DataFrame([{'Figura':p.stem.replace('_',' ').capitalize(), 'Archivo':str(p.relative_to(RAIZ)) if p.is_relative_to(RAIZ) else str(p)} for p in sorted(FIGURAS.glob('*.png'))])
display(catalogo)
(SALIDA/'catalogo_figuras.csv').parent.mkdir(parents=True,exist_ok=True)
catalogo.to_csv(SALIDA/'catalogo_figuras.csv',index=False)
resumen_final={
 'fecha':'2026-11-24', 'observaciones_entrenamiento':len(entrenamiento),
 'observaciones_prueba':len(prueba),'n_particiones':N_PARTICIONES,
 'mejor_individual':'XGBoost (profundidad 5, tasa 0.05)',
 'seleccion_nominal':'Regresión logística corregida con XGBoost',
 'auc_mejor_individual':0.9461201492641279,
 'auc_seleccionado':0.9462047549947218,
 'delta_auc':8.460573059387855e-05,
 'ic95_delta':[-0.00026258745106241544,0.00043972556755127727],
 'metodo_ci':'Bootstrap emparejado estratificado, 600 réplicas, condicional sobre OOF',
 'se_entrenaron_modelos_ahora':bool(EJECUTAR_PROTOCOLO_COMUN or EJECUTAR_BUSQUEDA_HIBRIDA),
 'hay_predicciones_privadas':bool(resultados_oof),
 'nota':'Se reutilizan resultados históricos; no equivale a validación independiente.'}
if resultados_comunes:
    resumen_final={'protocolo':'experiment021','seleccion_nominal':seleccionada,
                   'auc_seleccionado':float(resumen_comun.iloc[0].roc_auc),
                   'observaciones_entrenamiento':len(y),'observaciones_prueba':len(prueba),
                   'n_particiones':5,'se_entrenaron_modelos_ahora':bool(EJECUTAR_PROTOCOLO_COMUN),
                   'nota':'Selección por AUC OOF; no es prueba independiente.'}
(SALIDA/'resumen_final.json').write_text(json.dumps(resumen_final,indent=2,ensure_ascii=False),encoding='utf8')
print('Figuras:',len(catalogo),'| ruta:',FIGURAS.resolve())
print('Resultado: selección nominal por AUC OOF, sin ventaja estadística demostrada.')